# Combined Agent with Skills: UC-First + Genie Fallback + Lakebase Memory + Parallel MCP

This notebook extends `03_create_agent_definition.ipynb` by adding **Agent Skills** support.

**Prerequisite:** You must run `07-AgentSkillsGeneration.ipynb` first to generate skills via `optimize_anything` and persist them to Lakebase. The base agent created in `03_create_agent_definition.ipynb` does not include skills by default; this notebook produces `agent_with_skills.py` which loads skill metadata from Lakebase at startup.

The skills integration adds two things:
1. **Skill metadata** (name + description) is loaded from the Lakebase `skills` table at startup and appended to the system prompt
2. A **`load_skill` tool** lets the LLM read full skill instructions on demand from Lakebase

This notebook otherwise combines the same features from both agent architectures:

## Architecture

```
User Query → UC Functions (Parallel MCP) → Sufficiency Check → [If sufficient] → Response
                                                             ↓
                                                     [If partial/insufficient]
                                                             ↓
                                              Genie Fallback (unanswered parts only)
                                                             ↓
                                                   Synthesize → Response
```

## Features

### From UC-First Genie-Fallback:
- **UC functions tried FIRST** - deterministic, fast
- **Partial answer detection** - identifies what was/wasn't answered
- **Targeted Genie queries** - only asks about unanswered parts
- **Intelligent synthesis** - combines both responses seamlessly

### From Lakebase + MCP Agent:
- **Parallel tool execution** via MCP - all tools run simultaneously
- **Lakebase PostgreSQL memory** - conversation persistence across sessions
- **Connection pooling** - efficient database connections
- **OAuth credential caching** - 50-minute token cache

## Prerequisites

1. Run `00_setup` notebook first to create `config/atbat_assistant.json`
2. Run `07-AgentSkillsGeneration` to generate skill files to UC Volume **and** Lakebase
3. Ensure `genie` and `lakebase` sections are in the config
4. Lakebase instance configured (for memory and skills storage)

In [ ]:
%pip install -U -qqqq backoff databricks-openai uv databricks-agents "mlflow>=3.9" databricks-mcp langgraph-checkpoint-postgres psycopg[binary,pool] databricks-langchain langgraph
dbutils.library.restartPython()

## Load Configuration

Load configuration from `config/atbat_assistant.json` (created by setup notebook).

In [ ]:
# Load configuration from setup notebook
import json
import os
import base64
import gzip
from pathlib import Path
import mlflow

CONFIG = json.loads(Path("config/atbat_assistant.json").read_text())

# Extract configuration variables
PROMPT_NAME = CONFIG["prompt_registry"]["prompt_name"]
LLM_ENDPOINT_NAME = CONFIG["llm"]["endpoint_name"]
FINAL_SYNTHESIS_ENDPOINT_NAME = os.getenv(
    "ATBAT_FINAL_SYNTHESIS_ENDPOINT",
    CONFIG["llm"].get("final_synthesis_endpoint_name", "gpt-5-4-external"),
)
FINAL_SYNTHESIS_ENDPOINT_NAME = str(FINAL_SYNTHESIS_ENDPOINT_NAME or "").replace("databricks:/", "")
UC_MODEL_NAME = CONFIG["model"]["uc_model_name"]
MODEL_NAME = CONFIG["model"]["model_name"]
UC_TOOL_NAMES = CONFIG["tools"]["uc_tool_names"]
DISABLE_VECTOR_TOOLS = os.getenv("DISABLE_VECTOR_TOOLS", "true").lower() in {"1", "true", "yes"}
if DISABLE_VECTOR_TOOLS:
    UC_TOOL_NAMES = [
        tool_name for tool_name in UC_TOOL_NAMES
        if "embedding" not in tool_name.lower() and "vector" not in tool_name.lower()
    ]
    print(f"[CONFIG] Vector and embedding tools disabled; using {len(UC_TOOL_NAMES)} UC tools")
CATALOG = CONFIG["workspace"]["catalog"]
SCHEMA = CONFIG["workspace"]["schema"]

# Genie config
GENIE_SPACE_ID = CONFIG["genie"]["space_id"]
GENIE_NAME = CONFIG["genie"]["name"]

# Lakebase config
LAKEBASE_INSTANCE = CONFIG["lakebase"]["instance_name"]
LAKEBASE_HOST = CONFIG["lakebase"]["host"]

# Skills are generated by 07-AgentSkillsGeneration.ipynb and written to
# both a UC Volume and the Lakebase skills table. The agent loads them from
# Lakebase at runtime.
SKILLS_VOLUME_PATH = CONFIG["skills"]["gepa_volume_path"]

# Set MLflow experiment
EXPERIMENT_ID = CONFIG["mlflow"]["experiment_id"]
mlflow.set_experiment(experiment_id=EXPERIMENT_ID)

print(f"Loaded config from: config/atbat_assistant.json")
print(f"LLM Endpoint: {LLM_ENDPOINT_NAME}")
print(f"Final synthesis endpoint: {FINAL_SYNTHESIS_ENDPOINT_NAME or '(runtime model)'}")
print(f"UC Tools: {len(UC_TOOL_NAMES)} functions")
print(f"Genie Space: {GENIE_SPACE_ID}")
print(f"Lakebase Host: {LAKEBASE_HOST or '(not configured - memory disabled)'}")


## Load System Prompt

Load the system prompt from the MLflow Prompt Registry. The prompt should already be registered 
with a `production` alias from `03_create_agent_definition`. Skills will be appended to this 
prompt at agent startup.

In [ ]:
# Load the existing prompt from the registry (registered by 03_create_agent_definition)
system_prompt = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@production")
print(f"Loaded prompt '{PROMPT_NAME}' (version {system_prompt.version}, @production)")

## Setup Lakebase (Optional)

If you want conversation memory, set up Lakebase. Skip this cell if you don't need memory.

In [ ]:
# Only run if you have Lakebase configured and need to set up checkpoint tables
SETUP_LAKEBASE = False  # Set to True to create checkpoint tables

if SETUP_LAKEBASE:
    from langgraph.checkpoint.postgres import PostgresSaver
    import psycopg
    import uuid

    SP_CLIENT_ID = dbutils.secrets.get(scope=CONFIG["prompt_registry_auth"]["secret_scope_name"],
                                        key=CONFIG["prompt_registry_auth"]["oauth_client_id_key"])
    SP_CLIENT_SECRET = dbutils.secrets.get(scope=CONFIG["prompt_registry_auth"]["secret_scope_name"],
                                            key=CONFIG["prompt_registry_auth"]["oauth_client_secret_key"])

    from databricks.sdk import WorkspaceClient
    w = WorkspaceClient(
        host=CONFIG["prompt_registry_auth"]["databricks_host"],
        client_id=SP_CLIENT_ID,
        client_secret=SP_CLIENT_SECRET
    )

    cred = w.database.generate_database_credential(
        request_id=str(uuid.uuid4()),
        instance_names=[LAKEBASE_INSTANCE],
    )

    conn = psycopg.connect(
        f"dbname=databricks_postgres user={SP_CLIENT_ID} host={LAKEBASE_HOST} sslmode=require",
        password=cred.token
    )
    conn.autocommit = True

    checkpointer = PostgresSaver(conn)
    checkpointer.setup()
    print("Lakebase checkpoint tables created!")
    conn.close()
else:
    print("Skipping Lakebase setup (set SETUP_LAKEBASE = True to create checkpoint tables)")

## Define the Agent Code (with Skills)

Below we define the combined agent code in a single cell, enabling us to write it to a local Python file using the `%%writefile` magic command for subsequent logging and deployment.

This version adds **Agent Skills** support: skill metadata is loaded from Lakebase when available, with a UC Volume fallback for Free Edition environments where Lakebase is unavailable. A `load_skill` tool allows the LLM to read full skill content on demand.

### Agent Features:
- **Agent Skills** - Progressive disclosure of domain knowledge from Lakebase or UC Volumes
- **UC-first, Genie-fallback architecture** - Deterministic tools tried first
- **Parallel MCP tool execution** - All tool calls run simultaneously  
- **Partial answer detection** - Identifies what was/wasn't answered
- **Lakebase memory** - PostgreSQL-backed conversation persistence
- **Connection pooling** - Efficient database connections with OAuth caching

In [ ]:
%%writefile agent_with_skills.py
"""
Combined Agent with Skills: UC-First with Genie Fallback + Lakebase Memory + Parallel MCP Tool Calling

This agent combines:
1. UC functions tried FIRST via parallel MCP execution (deterministic, fast)
2. Sufficiency evaluation with partial answer detection
3. Genie fallback for unanswered parts (flexible, novel queries)
4. Lakebase PostgreSQL memory for conversation persistence
5. Connection pooling and OAuth credential caching
"""

import asyncio
import base64
import gzip
import json
import logging
import re
import os
import operator
import time
import uuid
from concurrent.futures import ThreadPoolExecutor, as_completed
from contextlib import contextmanager
from pathlib import Path
from threading import Lock
from typing import Annotated, Any, Generator, Literal, Optional, Sequence, TypedDict

import mlflow
import psycopg
from databricks.sdk import WorkspaceClient
from databricks.sdk.config import Config
from databricks_langchain import ChatDatabricks, UCFunctionToolkit
from databricks_langchain.genie import GenieAgent
from databricks_mcp import DatabricksMCPClient
from langchain_core.messages import AIMessage, AIMessageChunk, BaseMessage, HumanMessage, ToolMessage
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from mlflow.entities import SpanType
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
)
from psycopg.rows import dict_row
from psycopg_pool import ConnectionPool
from pydantic import BaseModel

logger = logging.getLogger(__name__)

########################################
# UTILITIES
########################################

def get_dbutils():
    """Get dbutils for secrets access."""
    try:
        from pyspark.dbutils import DBUtils
        from pyspark.sql import SparkSession
        spark = SparkSession.builder.getOrCreate()
        return DBUtils(spark)
    except (ImportError, Exception):
        try:
            import IPython
            ipython = IPython.get_ipython()
            if ipython and "dbutils" in ipython.user_ns:
                return ipython.user_ns["dbutils"]
        except:
            pass
    return None

dbutils = get_dbutils()

########################################
# CONFIGURATION
########################################

_CONFIG_PATH = Path("config/atbat_assistant.json")
if _CONFIG_PATH.exists():
    CONFIG = json.loads(_CONFIG_PATH.read_text())
else:
    config_env = os.getenv("ATBAT_ASSISTANT_CONFIG_JSON")
    if config_env:
        CONFIG = json.loads(config_env)
    else:
        raise FileNotFoundError(
            "config/atbat_assistant.json not found and ATBAT_ASSISTANT_CONFIG_JSON env var is not set"
        )

# Extract configuration values
PROMPT_NAME = CONFIG["prompt_registry"]["prompt_name"]
LLM_ENDPOINT_NAME = CONFIG["llm"]["endpoint_name"]
FINAL_SYNTHESIS_ENDPOINT_NAME = os.getenv(
    "ATBAT_FINAL_SYNTHESIS_ENDPOINT",
    CONFIG["llm"].get("final_synthesis_endpoint_name", "gpt-5-4-external"),
)
FINAL_SYNTHESIS_ENDPOINT_NAME = str(FINAL_SYNTHESIS_ENDPOINT_NAME or "").replace("databricks:/", "")
UC_TOOL_NAMES = CONFIG["tools"]["uc_tool_names"]
DISABLE_VECTOR_TOOLS = os.getenv("DISABLE_VECTOR_TOOLS", "true").lower() in {"1", "true", "yes"}
if DISABLE_VECTOR_TOOLS:
    UC_TOOL_NAMES = [
        tool_name for tool_name in UC_TOOL_NAMES
        if "embedding" not in tool_name.lower() and "vector" not in tool_name.lower()
    ]
    print(f"[CONFIG] Vector and embedding tools disabled; using {len(UC_TOOL_NAMES)} UC tools")
CATALOG = CONFIG["workspace"]["catalog"]
SCHEMA = CONFIG["workspace"]["schema"]

# Genie configuration
GENIE_SPACE_ID = CONFIG["genie"]["space_id"]
GENIE_NAME = CONFIG["genie"]["name"]

def _is_configured(value: Any) -> bool:
    return bool(value) and not str(value).startswith("PLACEHOLDER_")

SQL_WAREHOUSE_ID = CONFIG["genie"].get("warehouse_id") or os.getenv("DATABRICKS_SQL_WAREHOUSE_ID", "")
GENIE_ENABLED = _is_configured(GENIE_SPACE_ID)
if not GENIE_ENABLED:
    print("[CONFIG] Genie DISABLED because no Genie Space ID is configured")

# Auth configuration
SECRET_SCOPE_NAME = CONFIG["prompt_registry_auth"]["secret_scope_name"]
CLIENT_ID_KEY = CONFIG["prompt_registry_auth"]["oauth_client_id_key"]
CLIENT_SECRET_KEY = CONFIG["prompt_registry_auth"]["oauth_client_secret_key"]
DATABRICKS_HOST = CONFIG["prompt_registry_auth"]["databricks_host"]

# Set DATABRICKS_HOST if configured
if DATABRICKS_HOST and not os.getenv("DATABRICKS_HOST"):
    os.environ["DATABRICKS_HOST"] = DATABRICKS_HOST.rstrip("/")

# Load OAuth credentials from secrets into LOCAL variables (not env vars).
# The notebook runtime already sets DATABRICKS_TOKEN in the environment;
# putting OAuth creds there too causes a dual-auth conflict in the SDK.
_SP_CLIENT_ID = None
_SP_CLIENT_SECRET = None
if dbutils:
    try:
        _SP_CLIENT_ID = dbutils.secrets.get(scope=SECRET_SCOPE_NAME, key=CLIENT_ID_KEY).strip()
        _SP_CLIENT_SECRET = dbutils.secrets.get(scope=SECRET_SCOPE_NAME, key=CLIENT_SECRET_KEY).strip()
    except:
        pass

# Create WorkspaceClient: use OAuth M2M if available, otherwise auto-detect from env.
# Explicit auth_type prevents the SDK from discovering DATABRICKS_TOKEN in the
# environment and raising a dual-auth conflict.
if _SP_CLIENT_ID and _SP_CLIENT_SECRET:
    WORKSPACE_CLIENT = WorkspaceClient(config=Config(
        host=os.environ.get("DATABRICKS_HOST"),
        client_id=_SP_CLIENT_ID,
        client_secret=_SP_CLIENT_SECRET,
        auth_type="oauth-m2m",
    ))
else:
    WORKSPACE_CLIENT = WorkspaceClient()

# Ensure DATABRICKS_HOST is set
if not os.environ.get("DATABRICKS_HOST") and WORKSPACE_CLIENT.config.host:
    os.environ["DATABRICKS_HOST"] = WORKSPACE_CLIENT.config.host.rstrip("/")

# MLflow setup
if os.environ.get("DATABRICKS_HOST") and not os.environ.get("MLFLOW_TRACKING_URI"):
    os.environ["MLFLOW_TRACKING_URI"] = "databricks"

mlflow.set_registry_uri("databricks-uc")
MLFLOW_EXPERIMENT_ID = CONFIG["mlflow"]["experiment_id"]
if MLFLOW_EXPERIMENT_ID:
    try:
        mlflow.set_experiment(experiment_id=MLFLOW_EXPERIMENT_ID)
        logger.info(f"MLflow experiment set to {MLFLOW_EXPERIMENT_ID}")
    except Exception as e:
        logger.warning(f"Could not set MLflow experiment: {e}")

PROMPT_TEMPLATE_FALLBACK = """You are a hitting assistant tasked with helping batters prepare for matchups against specific pitchers.

Use deterministic matchup history, pitcher tendency, arsenal, roster, lineup, and Genie-backed table analysis to answer questions.

The team abbreviations to choose from are below, use the 3 letter acronyms to get data:

Teams:TEX,CHC,LAA,LAD,STL,PHI,ARI,OAK,TBR,MIN,CLE,CHW,NYM,COL,SEA,MIA,SDP,WSN,HOU,SFG,CIN,BAL,KCR,PIT,ATL,NYY,DET,MIL,TOR,BOS,ATH

General rules:

- Always assume the most recent season (2025) if a season is not provided.
- Always leverage the tooling you have available to answer user queries.
- Only perform the minimum necessary tool calls to complete a request. Do not exceed 8 tool calls before providing a response.
- If you need multiple tools, include all tool_calls in a single assistant message; don't chain them one-by-one.
- Before calling a tool, include every required argument. Default season_year to 2025 when missing. For runner-state tendency tools, set unspecified base occupancy flags to false and use a 0-0 count when the user did not specify a count.
- If a tool times out, returns a 504, or returns empty rows, do not end with an apology. Use alternate available evidence when possible and provide a next-best coaching plan.

For open-ended requests always respond with the following format in markdown:
# At-Bat Assistant Assessment
## Data collected
- Summarize the data collected to inform the analysis in a very concise fashion. You do not need reference the tools by their explicit name, just need to summarize the data collected. Ex. Collected data on tendencies by count. Do not exceed 50 words
## Pitcher Approach
- Summarize how the pitcher might approach the batter. Use discretion to include further subheadings by count or scenario, or just include it all under the pitcher approach heading if that is not necessary. Do not exceed 200 words
## Recommendation
- Summarize how the batter should approach potential at-bats(s) in a concise format, 50-75 words.

Fallback rules:
- If exact data is unavailable, still provide a grounded action plan with: what was checked, what was unavailable, and how the hitter should adjust using pitch mix, handedness, count, location, and runner-state principles.
- Do not claim the baseball data does not exist unless all available retrieval paths have failed or returned no rows.

CONVERSATION HISTORY: You have access to previous messages in this conversation thread.
- ONLY reference prior conversation context if the user's current question is ambiguous or explicitly refers to something discussed earlier.
- If the user asks a NEW, self-contained question, answer it directly using your tools WITHOUT referencing prior context.
- Do NOT proactively bring up previous topics or assume the user wants comparisons to earlier queries.
"""

# Load system prompt
try:
    PROMPT_URI_AGENT = f"prompts:/{PROMPT_NAME}@production"
    SYSTEM_PROMPT = mlflow.genai.load_prompt(PROMPT_URI_AGENT)
except Exception as e:
    SYSTEM_PROMPT = PROMPT_TEMPLATE_FALLBACK
    logger.warning(f"Could not load system prompt from registry, using embedded fallback: {e}")

# Lakebase is disabled for Free Edition. Skills load from UC Volumes.
DISABLE_LAKEBASE = True

if DISABLE_LAKEBASE:
    print("[CONFIG] Lakebase DISABLED for Free Edition deployment")
    LAKEBASE_CONFIG = {"instance_name": "", "conn_host": ""}
else:
    LAKEBASE_CONFIG = {
        "instance_name": CONFIG["lakebase"]["instance_name"],
        "conn_host": CONFIG["lakebase"]["conn_host"],
        "conn_db_name": "databricks_postgres",
        "conn_ssl_mode": "require",
    }
    print(f"[CONFIG] Lakebase instance: {LAKEBASE_CONFIG['instance_name']}")
    print(f"[CONFIG] Lakebase host: {LAKEBASE_CONFIG['conn_host'] or '(not configured)'}")

def _content_to_text(content: Any) -> str:
    """Convert LangChain/OpenAI content blocks to user-visible text."""
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict):
                if item.get("type") == "reasoning":
                    continue
                parts.append(str(item.get("text") or item.get("content") or ""))
            elif hasattr(item, "text"):
                parts.append(str(item.text))
            else:
                parts.append(str(item))
        return "".join(parts)
    return str(content)


def _strip_reasoning_artifacts(content: Any) -> str:
    """Remove GPT OSS reasoning blocks that can leak into text content."""
    text = _content_to_text(content)
    decoder = json.JSONDecoder()
    cleaned = []
    remaining = text

    while remaining:
        stripped = remaining.lstrip()
        if stripped.startswith("["):
            try:
                obj, end = decoder.raw_decode(stripped)
                if isinstance(obj, list) and obj and isinstance(obj[0], dict) and obj[0].get("type") == "reasoning":
                    remaining = stripped[end:]
                    continue
            except Exception:
                pass

        cleaned.append(remaining[0])
        remaining = remaining[1:]

    return "".join(cleaned).strip()

########################################
# SKILL QUERIES
########################################

SKILLS_VOLUME_PATH = CONFIG.get("skills", {}).get("gepa_volume_path", "")
_ENV_SKILL_ENTRIES = []
try:
    if os.getenv("ATBAT_SKILLS_B64"):
        _ENV_SKILL_ENTRIES = json.loads(gzip.decompress(base64.b64decode(os.getenv("ATBAT_SKILLS_B64"))).decode("utf-8"))
    else:
        _ENV_SKILL_ENTRIES = json.loads(os.getenv("ATBAT_SKILLS_JSON", "[]"))
except Exception as e:
    logger.warning(f"Could not parse ATBAT_SKILLS_JSON: {e}")
_skill_db_cache = {"token": None, "timestamp": 0, "username": None}
_skill_source = "none"


def _get_skill_db_conn():
    """Return a short-lived psycopg connection for skill table queries."""
    now = time.time()
    if now - _skill_db_cache["timestamp"] > 3000:
        cred = WORKSPACE_CLIENT.database.generate_database_credential(
            request_id=str(uuid.uuid4()),
            instance_names=[LAKEBASE_CONFIG["instance_name"]],
        )
        _skill_db_cache["token"] = cred.token
        _skill_db_cache["timestamp"] = now
    if _skill_db_cache["username"] is None:
        _skill_db_cache["username"] = WORKSPACE_CLIENT.current_user.me().user_name

    return psycopg.connect(
        f"dbname={LAKEBASE_CONFIG.get('conn_db_name', 'databricks_postgres')} "
        f"user={_skill_db_cache['username']} "
        f"host={LAKEBASE_CONFIG['conn_host']} sslmode=require",
        password=_skill_db_cache["token"],
        autocommit=True,
        row_factory=dict_row,
    )


def _parse_skill_frontmatter(content: str) -> dict[str, str]:
    """Extract skill name and description from YAML frontmatter."""
    match = re.match(r"^---\s*\n(.*?)\n---", content, re.DOTALL)
    if not match:
        return {}
    frontmatter = match.group(1)
    result = {}
    for field in ["name", "description"]:
        field_match = re.search(rf"^{field}:\s*>?\s*\n?(.*?)(?=\n\w|\Z)", frontmatter, re.MULTILINE | re.DOTALL)
        if field_match:
            result[field] = " ".join(field_match.group(1).strip().split())
    return result


def _query_env_skills_metadata():
    rows = []
    for entry in _ENV_SKILL_ENTRIES:
        filename = entry.get("filename", "")
        content = entry.get("content", "")
        if not filename.endswith("skill.md"):
            continue
        frontmatter = _parse_skill_frontmatter(content)
        if not frontmatter.get("name"):
            continue
        rows.append({
            "name": frontmatter.get("name", Path(filename).parent.name),
            "description": frontmatter.get("description", ""),
            "filename": filename,
        })
    return rows


def _query_env_skill_content(filename):
    folder = str(Path(filename).parent)
    parts = []
    for suffix in ("skill.md", "gotcha.md", "examples.md"):
        target = f"{folder}/{suffix}"
        for entry in _ENV_SKILL_ENTRIES:
            if entry.get("filename") == target:
                parts.append(entry.get("content", ""))
                break
    if not parts:
        return f"Skill '{filename}' not found in packaged skills."
    parts.append(
        "\n\n## Runtime Data Guardrail\n"
        "- CRITICAL: If only an arsenal lookup returned pitch types, do not infer usage rates, sequencing, speeds, movement, or a generic MLB distribution.\n"
        "- For pitch-type inventory questions, list only returned pitch types and explicitly state that usage/tendency data was not requested or not available.\n"
        "- Do not add percentages or likely count plans unless a tendency or matchup tool returned that evidence."
    )
    return "\n\n".join(parts)


def _query_lakebase_skills_metadata():
    conn = _get_skill_db_conn()
    try:
        with conn.cursor() as cur:
            cur.execute("SELECT name, description, filename FROM skills ORDER BY name")
            return cur.fetchall()
    finally:
        conn.close()


def _query_lakebase_skill_content(filename):
    conn = _get_skill_db_conn()
    try:
        with conn.cursor() as cur:
            cur.execute("SELECT content FROM skills WHERE filename = %s", (filename,))
            row = cur.fetchone()
            return row["content"] if row else f"Skill '{filename}' not found in database."
    finally:
        conn.close()


def _query_volume_skills_metadata():
    """Load skill metadata from markdown files in the configured UC Volume."""
    if not SKILLS_VOLUME_PATH:
        return []
    base = Path(SKILLS_VOLUME_PATH)
    if not base.exists():
        return []
    rows = []
    skill_files = list(base.glob("*.md")) + list(base.glob("*/skill.md"))
    for path in sorted(skill_files):
        content = path.read_text()
        frontmatter = _parse_skill_frontmatter(content)
        if not frontmatter.get("name"):
            continue
        filename = str(path.relative_to(base))
        rows.append({
            "name": frontmatter.get("name", path.parent.name if path.name == "skill.md" else path.stem),
            "description": frontmatter.get("description", ""),
            "filename": filename,
        })
    return rows


def _query_volume_skill_content(filename):
    if not SKILLS_VOLUME_PATH:
        return f"Skill '{filename}' not found because SKILLS_VOLUME_PATH is not configured."
    path = Path(SKILLS_VOLUME_PATH) / filename
    if not path.exists():
        return f"Skill '{filename}' not found in volume {SKILLS_VOLUME_PATH}."
    if path.name == "skill.md":
        parts = [path.read_text()]
        for sibling in ("gotcha.md", "examples.md"):
            sibling_path = path.parent / sibling
            if sibling_path.exists():
                parts.append(sibling_path.read_text())
        parts.append(
            "\n\n## Runtime Data Guardrail\n"
            "- CRITICAL: If only an arsenal lookup returned pitch types, do not infer usage rates, sequencing, speeds, movement, or a generic MLB distribution.\n"
            "- For pitch-type inventory questions, list only returned pitch types and explicitly state that usage/tendency data was not requested or not available.\n"
            "- Do not add percentages or likely count plans unless a tendency or matchup tool returned that evidence."
        )
        return "\n\n".join(parts)
    return path.read_text()


def _query_skills_metadata():
    """Load skill metadata from markdown files in the configured UC Volume."""
    global _skill_source
    rows = _query_env_skills_metadata()
    if rows:
        _skill_source = "environment"
        return rows
    rows = _query_volume_skills_metadata()
    if rows:
        _skill_source = "UC Volume"
        return rows
    _skill_source = "none"
    return []


def _query_skill_content(filename):
    """Load skill content from the source that provided metadata."""
    if _skill_source == "environment":
        return _query_env_skill_content(filename)
    if _skill_source == "UC Volume":
        return _query_volume_skill_content(filename)
    return f"Skill '{filename}' not found because no skill source is configured."


# Load skill metadata at startup.
SKILL_METADATA_BLOCK = ""
try:
    _skill_rows = _query_skills_metadata()
    if _skill_rows:
        _skills_xml = []
        _genie_skill_count = 0
        for _s in _skill_rows:
            _name_lower = _s["name"].lower()
            _desc_lower = _s["description"].lower()
            if any(term in _name_lower or term in _desc_lower for term in ("similar", "embedding", "vector")):
                logger.info(f"Comparison skill excluded from prompt metadata: {_s['name']}")
                continue
            if "genie" in _name_lower or "genie" in _desc_lower:
                _genie_skill_count += 1
                logger.info(f"Genie skill excluded from prompt metadata: {_s['name']}")
                continue
            _skills_xml.append(
                f'<skill>\n  <name>{_s["name"]}</name>\n'
                f'  <description>{_s["description"]}</description>\n'
                f'  <filename>{_s["filename"]}</filename>\n</skill>'
            )
        if _genie_skill_count:
            logger.info(f"Excluded {_genie_skill_count} Genie skill(s) (auto-loaded in fallback path)")
        if GENIE_ENABLED:
            _genie_routing_block = (
                "GENIE ROUTING (IMPORTANT):\n"
                "- Some skills reference `genie_space_query` or `sufficiency_eval`. "
                "These are NOT tools you call directly. They are handled automatically "
                "by the system after your UC tool calls are evaluated for sufficiency.\n"
                "- Your job: use the available UC tools to answer as much as possible. "
                "If a query is outside the scope of your UC tools, clearly state what "
                "you found and what remains unanswered. The system will automatically "
                "route unanswered portions to a data exploration service (Genie).\n"
                "- NEVER say you cannot answer because genie_space_query is unavailable. "
                "Instead, attempt the query with your available tools and let the system "
                "handle the fallback."
            )
        else:
            _genie_routing_block = (
                "GENIE ROUTING (DISABLED):\n"
                "- Genie is not configured in this deployment. Use UC functions and "
                "UC functions where available. If those tools cannot answer "
                "the full query, say what was answered and what requires a Genie Space."
            )

        SKILL_METADATA_BLOCK = (
            "\n\n<available_skills>\n"
            + "\n".join(_skills_xml)
            + "\n</available_skills>\n\n"
            "SKILL LOADING INSTRUCTIONS:\n"
            "- ALWAYS call load_skill with the matching skill's filename BEFORE "
            "calling any UC functions. Read the full skill instructions first, "
            "then use them to guide your tool selection, workflow, and response "
            "formatting.\n"
            "- If multiple skills could match, load the most specific one.\n"
            "- Do NOT skip this step. Skills contain critical workflow guidance, "
            "fallback paths, and quality checklists that you cannot infer alone.\n"
            "- After completing your response, verify it satisfies the skill's "
            "'Before Responding' checklist before returning.\n\n"
            + _genie_routing_block
        )
        logger.info(f"Loaded {len(_skill_rows)} skill(s) from {_skill_source}")
    else:
        logger.info("No skills found in UC Volume; skills disabled")
except Exception as e:
    _skill_rows = []
    logger.warning(f"Could not load skills: {e}")

########################################
# MCP PARALLEL EXECUTION
########################################

# Cache auth for per-thread MCP clients (reuse the same local SP creds)
_CACHED_HOST = os.environ.get("DATABRICKS_HOST")
_CACHED_TOKEN = os.environ.get("DATABRICKS_TOKEN")
_FUNCTION_INFO_CACHE: dict[str, Any] = {}


def _sql_literal(value: Any) -> str:
    if value is None:
        return "NULL"
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, (int, float)):
        return str(value)
    escaped = str(value).replace("'", "''")
    return f"'{escaped}'"


def _get_sql_warehouse_id() -> str:
    if SQL_WAREHOUSE_ID:
        return SQL_WAREHOUSE_ID
    warehouses = list(WORKSPACE_CLIENT.warehouses.list())
    for warehouse in warehouses:
        if getattr(warehouse, "enable_serverless_compute", False):
            return warehouse.id
    if warehouses:
        return warehouses[0].id
    raise RuntimeError("No SQL warehouse available for UC function execution")


def _get_function_info(full_name: str):
    if full_name not in _FUNCTION_INFO_CACHE:
        _FUNCTION_INFO_CACHE[full_name] = WORKSPACE_CLIENT.functions.get(full_name)
    return _FUNCTION_INFO_CACHE[full_name]


_DEFAULT_TOOL_ARGS = {
    "season_year": 2025,
    "b": 0,
    "s": 0,
    "p_on_1b": False,
    "p_on_2b": False,
    "p_on_3b": False,
}

_TRANSIENT_TOOL_ERROR_MARKERS = (
    "504 gateway timeout",
    "execution timed out after 90 seconds",
    "provisioning resources for function execution",
    "temporarily unavailable",
    "timed out",
    "read timed out",
)


def _is_transient_tool_error(exc: BaseException) -> bool:
    text = str(exc).lower()
    return any(marker in text for marker in _TRANSIENT_TOOL_ERROR_MARKERS)


def _normalize_tool_args(tool_name: str, args: dict | None) -> tuple[dict, list[str]]:
    normalized = dict(args or {})
    mcp_tool_name = _resolve_tool_name(tool_name.replace(".", "__") if "." in tool_name else tool_name)
    full_name = mcp_tool_name.replace("__", ".")

    try:
        function_info = _get_function_info(full_name)
        parameters = list(getattr(getattr(function_info, "input_params", None), "parameters", []) or [])
    except Exception:
        parameters = []

    missing_required = []
    for param in sorted(parameters, key=lambda p: p.position or 0):
        name = param.name
        if name in normalized and normalized[name] is not None:
            continue
        if name in _DEFAULT_TOOL_ARGS:
            normalized[name] = _DEFAULT_TOOL_ARGS[name]
        else:
            missing_required.append(name)

    return normalized, missing_required


def _execute_uc_function_with_retries(tool_name: str, args: dict | None) -> str:
    normalized_args, missing_required = _normalize_tool_args(tool_name, args)
    if missing_required:
        missing = ", ".join(missing_required)
        raise ValueError(f"Missing required tool parameter(s) for {tool_name}: {missing}")

    max_attempts = int(os.getenv("UC_TOOL_MAX_ATTEMPTS", "2"))
    base_sleep = float(os.getenv("UC_TOOL_RETRY_BASE_SLEEP_SECONDS", "1.5"))
    last_error = None
    for attempt in range(max_attempts):
        try:
            return _execute_uc_function_mcp(tool_name, normalized_args)
        except Exception as exc:
            last_error = exc
            if attempt >= max_attempts - 1 or not _is_transient_tool_error(exc):
                raise
            sleep_seconds = base_sleep * (2 ** attempt)
            logger.warning(
                f"Retrying transient UC tool failure for {tool_name} "
                f"in {sleep_seconds:.1f}s ({attempt + 1}/{max_attempts}): {exc}"
            )
            time.sleep(sleep_seconds)

    raise last_error or RuntimeError(f"UC tool execution failed for {tool_name}")


def _format_statement_result(resp: Any) -> str:
    columns = []
    try:
        columns = [col.name for col in resp.manifest.schema.columns]
    except Exception:
        pass

    rows = []
    try:
        rows = resp.result.data_array or []
    except Exception:
        rows = []

    truncated = False
    try:
        truncated = bool(resp.manifest.truncated)
    except Exception:
        pass

    return json.dumps({
        "is_truncated": truncated,
        "columns": columns,
        "rows": rows,
    })


def _execute_uc_function_sql(tool_name: str, args: dict) -> str:
    mcp_tool_name = _resolve_tool_name(tool_name.replace(".", "__") if "." in tool_name else tool_name)
    full_name = mcp_tool_name.replace("__", ".")
    function_info = _get_function_info(full_name)
    parameters = list(getattr(getattr(function_info, "input_params", None), "parameters", []) or [])
    ordered_names = [p.name for p in sorted(parameters, key=lambda p: p.position or 0)]

    named_args = []
    for name in ordered_names:
        if name in args:
            named_args.append(f"{name} => {_sql_literal(args[name])}")
    if not named_args:
        for name, value in args.items():
            named_args.append(f"{name} => {_sql_literal(value)}")

    invocation = f"{full_name}({', '.join(named_args)})"
    data_type = str(getattr(function_info, "data_type", "")).upper()
    statement = f"SELECT * FROM {invocation} LIMIT 100" if "TABLE_TYPE" in data_type else f"SELECT {invocation} AS output"

    timeout_seconds = int(os.getenv("UC_SQL_TOOL_TIMEOUT_SECONDS", "120"))
    deadline = time.time() + timeout_seconds
    resp = WORKSPACE_CLIENT.statement_execution.execute_statement(
        warehouse_id=_get_sql_warehouse_id(),
        statement=statement,
        wait_timeout="30s",
    )

    while str(resp.status.state).endswith("PENDING") or str(resp.status.state).endswith("RUNNING"):
        if time.time() >= deadline:
            try:
                WORKSPACE_CLIENT.statement_execution.cancel_execution(resp.statement_id)
            except Exception:
                pass
            raise TimeoutError(f"SQL function execution timed out after {timeout_seconds}s: {full_name}")
        time.sleep(2)
        resp = WORKSPACE_CLIENT.statement_execution.get_statement(resp.statement_id)

    state = str(resp.status.state)
    if not state.endswith("SUCCEEDED"):
        error = getattr(resp.status, "error", None)
        raise RuntimeError(f"SQL function execution failed for {full_name}: {state} {error}")

    return _format_statement_result(resp)


def _execute_uc_function_mcp(tool_name: str, args: dict) -> str:
    """Execute a UC function using Databricks MCP with per-thread client isolation."""
    if os.getenv("USE_MCP_FUNCTION_EXECUTION", "false").lower() not in {"1", "true", "yes"}:
        return _execute_uc_function_sql(tool_name, args)
    host = os.environ.get("DATABRICKS_HOST", "").rstrip("/")
    if not host:
        host = WORKSPACE_CLIENT.config.host.rstrip("/")
    mcp_server_url = f"{host}/api/2.0/mcp/functions/{CATALOG}/{SCHEMA}"

    mcp_tool_name = _resolve_tool_name(tool_name.replace(".", "__") if "." in tool_name else tool_name)

    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

    try:
        # Create per-thread client — explicit auth_type avoids dual-auth
        # conflicts when the notebook runtime re-injects DATABRICKS_TOKEN.
        if _SP_CLIENT_ID and _SP_CLIENT_SECRET:
            thread_workspace_client = WorkspaceClient(config=Config(
                host=_CACHED_HOST,
                client_id=_SP_CLIENT_ID,
                client_secret=_SP_CLIENT_SECRET,
                auth_type="oauth-m2m",
            ))
        elif _CACHED_TOKEN:
            thread_workspace_client = WorkspaceClient(config=Config(
                host=_CACHED_HOST,
                token=_CACHED_TOKEN,
                auth_type="pat",
            ))
        else:
            thread_workspace_client = WorkspaceClient()

        mcp_client = DatabricksMCPClient(
            server_url=mcp_server_url,
            workspace_client=thread_workspace_client
        )

        result = mcp_client.call_tool(mcp_tool_name, args)

        if hasattr(result, 'content'):
            if isinstance(result.content, list):
                return "\n".join(
                    str(item.text if hasattr(item, 'text') else item)
                    for item in result.content
                )
            return str(result.content)
        return str(result)

    except Exception as e:
        logger.error(f"MCP call failed for {tool_name}: {e}")
        raise
    finally:
        loop.close()


########################################
# LAKEBASE CONNECTION POOLING
########################################

class CredentialConnection(psycopg.Connection):
    """Custom connection class with OAuth token caching."""

    workspace_client = None
    instance_name = None
    _cached_credential = None
    _cache_timestamp = None
    _cache_duration = 3000  # 50 minutes
    _cache_lock = Lock()

    @classmethod
    def connect(cls, conninfo='', **kwargs):
        if cls.workspace_client is None or cls.instance_name is None:
            raise ValueError("workspace_client and instance_name must be set")
        kwargs['password'] = cls._get_cached_credential()
        return super().connect(conninfo, **kwargs)

    @classmethod
    def _get_cached_credential(cls):
        with cls._cache_lock:
            current_time = time.time()
            if (cls._cached_credential is not None and
                cls._cache_timestamp is not None and
                current_time - cls._cache_timestamp < cls._cache_duration):
                return cls._cached_credential

            credential = cls.workspace_client.database.generate_database_credential(
                request_id=str(uuid.uuid4()),
                instance_names=[cls.instance_name]
            )
            cls._cached_credential = credential.token
            cls._cache_timestamp = current_time
            return cls._cached_credential


########################################
# GENIE CONFIGURATION
########################################

class GenieConfig(BaseModel):
    space_id: str
    name: str
    description: str = ""
    max_retries: int = 2


GENIE_CONFIG = GenieConfig(
    space_id=GENIE_SPACE_ID,
    name=GENIE_NAME,
    description="""This agent is a FALLBACK for when UC functions cannot answer the question.
The Genie space contains comprehensive baseball Statcast pitch-level data including:
- Pitch characteristics (velocity, spin, movement, release point)
- Batter outcomes (launch speed, angle, hit distance, wOBA)
- Player and team information
- Historical matchup data
Use this for flexible, novel queries that predefined UC functions cannot handle.

SQL generation guardrails:
- Generate Databricks SQL syntax only.
- For zipping arrays, use arrays_zip(...) with an s. Do not use array_zip(...).
- Prefer simple GROUP BY, CASE WHEN, avg, count, sum, and percentile calculations over complex array transformations.
- If exploding arrays is required, use posexplode or explode with valid Databricks SQL syntax.
- Query only tables attached to the Genie space.""",
    max_retries=2,
)


########################################
# AGENT STATE
########################################

class AgentState(TypedDict):
    """State for the combined agent with memory and Genie fallback."""
    messages: Annotated[Sequence[BaseMessage], add_messages]
    original_query: str
    uc_response: str | None
    uc_answered_parts: str | None
    unanswered_parts: str | None
    uc_sufficient: bool
    genie_response: str | None
    genie_error: str | None
    final_response: str | None
    tool_calls_made: Annotated[list[dict[str, Any]], operator.add]
    custom_inputs: Optional[dict[str, Any]]
    custom_outputs: Optional[dict[str, Any]]


########################################
# LOAD UC TOOLS
########################################

tools = []
if UC_TOOL_NAMES:
    try:
        print(f"[STARTUP] Loading {len(UC_TOOL_NAMES)} UC tools...")
        uc_toolkit = UCFunctionToolkit(function_names=UC_TOOL_NAMES)
        tools.extend(uc_toolkit.tools)
        print(f"[STARTUP] UC tools loaded successfully: {len(tools)} tools")
    except Exception as e:
        print(f"[STARTUP ERROR] Failed to load UC toolkit: {e}")
        logger.error(f"Failed to load UC toolkit: {e}")
else:
    print("[STARTUP] No UC tools configured")

# Add skill loader tool if skills are available
if SKILL_METADATA_BLOCK:
    from langchain_core.tools import tool as _lc_tool

    @_lc_tool
    def load_skill(filename: str) -> str:
        """Load detailed instructions for a specific agent skill from the UC Volume.
        Call this when a user query matches one of the available skills
        listed in the system prompt. Pass the filename exactly as shown in the
        available_skills block."""
        with mlflow.start_span(name="load_skill", span_type=SpanType.RETRIEVER) as span:
            span.set_inputs({"filename": filename})
            content = _query_skill_content(filename)
            span.set_outputs({"content_length": len(content), "filename": filename})
            return content

    tools.append(load_skill)
    print(f"[STARTUP] load_skill tool registered ({len(_skill_rows)} skills from UC Volume)")


# Build lookup for resolving potentially truncated tool names from LLM
_TOOL_NAME_SUFFIX_MAP = {}
for _tn in UC_TOOL_NAMES:
    # Full dotted name -> underscored MCP name
    _mcp_name = _tn.replace(".", "__")
    _TOOL_NAME_SUFFIX_MAP[_mcp_name] = _mcp_name
    # Also map by just the function name (last segment)
    _func_name = _tn.split(".")[-1]
    _TOOL_NAME_SUFFIX_MAP[_func_name] = _mcp_name


def _resolve_tool_name(name: str) -> str:
    """Resolve a potentially truncated tool name to the correct full MCP name.
    
    The LLM sometimes truncates long catalog names (e.g. 'cmegdemos_catalog' -> 'mos_catalog').
    This resolves by matching the function suffix.
    """
    if name in _TOOL_NAME_SUFFIX_MAP:
        return _TOOL_NAME_SUFFIX_MAP[name]
    # Try matching by suffix (function name after last __)
    parts = name.split("__")
    func_suffix = parts[-1] if parts else name
    if func_suffix in _TOOL_NAME_SUFFIX_MAP:
        resolved = _TOOL_NAME_SUFFIX_MAP[func_suffix]
        logger.info(f"Resolved truncated tool name '{name}' -> '{resolved}'")
        return resolved
    # Try endswith matching
    for full_name in _TOOL_NAME_SUFFIX_MAP.values():
        if full_name.endswith(func_suffix):
            logger.info(f"Resolved truncated tool name '{name}' -> '{full_name}'")
            return full_name
    return name


########################################
# MAIN AGENT CLASS
########################################

class CombinedLangGraphAgent(ResponsesAgent):
    """
    Combined agent with:
    - UC-first, Genie-fallback architecture
    - Parallel MCP tool execution
    - Lakebase PostgreSQL memory
    - Partial answer detection
    """

    def __init__(self, lakebase_config: dict[str, Any], genie_config: GenieConfig):
        self.lakebase_config = lakebase_config
        self.genie_config = genie_config
        self.workspace_client = WORKSPACE_CLIENT

        # LLM setup
        self.model = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)
        self.synthesis_model = None
        if FINAL_SYNTHESIS_ENDPOINT_NAME and FINAL_SYNTHESIS_ENDPOINT_NAME != LLM_ENDPOINT_NAME:
            try:
                self.synthesis_model = ChatDatabricks(endpoint=FINAL_SYNTHESIS_ENDPOINT_NAME)
                logger.info(f"Using {FINAL_SYNTHESIS_ENDPOINT_NAME} for final synthesis")
            except Exception as e:
                logger.warning(f"Could not initialize final synthesis endpoint {FINAL_SYNTHESIS_ENDPOINT_NAME}: {e}")
        self.system_prompt = SYSTEM_PROMPT
        self.model_with_tools = self.model.bind_tools(tools) if tools else self.model

        # Genie agent
        self.genie_agent = None
        if _is_configured(genie_config.space_id):
            self.genie_agent = GenieAgent(
                genie_space_id=genie_config.space_id,
                genie_agent_name=genie_config.name,
                description=genie_config.description,
            )
        else:
            logger.info("Genie disabled because no Genie Space ID is configured")

        # Connection pool settings
        self.pool_min_size = int(os.getenv("DB_POOL_MIN_SIZE", "1"))
        self.pool_max_size = int(os.getenv("DB_POOL_MAX_SIZE", "10"))
        self.pool_timeout = float(os.getenv("DB_POOL_TIMEOUT", "30.0"))

        cache_duration_minutes = int(os.getenv("DB_TOKEN_CACHE_MINUTES", "50"))
        CredentialConnection._cache_duration = cache_duration_minutes * 60

        # Only create pool if Lakebase is configured
        self._connection_pool = None
        if self.lakebase_config.get("conn_host"):
            try:
                self._connection_pool = self._create_rotating_pool()
            except Exception as e:
                logger.warning(f"Failed to create Lakebase connection pool: {e}")

        mlflow.langchain.autolog()

    def _get_username(self) -> str:
        try:
            sp = self.workspace_client.current_service_principal.me()
            return sp.application_id
        except Exception:
            user = self.workspace_client.current_user.me()
            return user.user_name

    def _create_rotating_pool(self) -> ConnectionPool:
        CredentialConnection.workspace_client = self.workspace_client
        CredentialConnection.instance_name = self.lakebase_config["instance_name"]

        username = self._get_username()
        host = self.lakebase_config["conn_host"]
        database = self.lakebase_config.get("conn_db_name", "databricks_postgres")

        pool = ConnectionPool(
            conninfo=f"dbname={database} user={username} host={host} sslmode=require",
            connection_class=CredentialConnection,
            min_size=self.pool_min_size,
            max_size=self.pool_max_size,
            timeout=self.pool_timeout,
            open=True,
            check=ConnectionPool.check_connection,
            kwargs={
                "autocommit": True,
                "row_factory": dict_row,
                "keepalives": 1,
                "keepalives_idle": 60,
                "keepalives_interval": 10,
                "keepalives_count": 5,
            }
        )

        # Test connection
        with pool.connection() as conn:
            with conn.cursor() as cursor:
                cursor.execute("SELECT 1")

        return pool

    @contextmanager
    def get_connection(self):
        """Get a validated connection from the pool with retry logic."""
        if not self._connection_pool:
            yield None
            return

        max_retries = 3
        retry_count = 0

        while retry_count < max_retries:
            try:
                with self._connection_pool.connection() as conn:
                    with conn.cursor() as cursor:
                        cursor.execute("SELECT 1")
                    yield conn
                    return
            except (psycopg.OperationalError, psycopg.InterfaceError) as e:
                retry_count += 1
                if retry_count >= max_retries:
                    self._connection_pool = self._create_rotating_pool()
                    with self._connection_pool.connection() as conn:
                        yield conn
                        return
                time.sleep(0.5 * retry_count)

    def _parallel_tool_node(self, state: AgentState) -> dict:
        """Execute all tool calls in parallel using MCP with MLflow tracing.

        Skill-first prioritization: if any tool call is load_skill, execute ONLY
        those calls and return early. The LLM will see the skill instructions
        before deciding which UC functions to call on the next turn.
        """
        messages = state["messages"]
        last_message = messages[-1] if messages else None

        if not isinstance(last_message, AIMessage) or not last_message.tool_calls:
            return {"messages": [], "tool_calls_made": []}

        tool_calls = last_message.tool_calls

        # Separate load_skill calls from other tool calls
        skill_calls = [tc for tc in tool_calls if tc["name"] == "load_skill"]
        other_calls = [tc for tc in tool_calls if tc["name"] != "load_skill"]

        # If there are skill calls, execute ONLY those first so the LLM reads
        # the skill instructions before issuing UC function calls.
        # We must still return ToolMessage placeholders for the deferred calls
        # because the API requires every tool_use to have a matching tool_result.
        _deferred_calls = []
        if skill_calls and other_calls:
            logger.info(
                f"Skill-first: executing {len(skill_calls)} load_skill call(s), "
                f"deferring {len(other_calls)} other tool call(s)"
            )
            _deferred_calls = other_calls
            tool_calls = skill_calls

        start_time = time.time()

        tool_results = []
        max_workers = int(os.getenv("UC_TOOL_MAX_WORKERS", "4"))
        with ThreadPoolExecutor(max_workers=min(len(tool_calls), max_workers)) as executor:
            futures = {}
            tool_start_times = {}
            for tc in tool_calls:
                tool_start_times[tc["id"]] = time.time()
                # Route local tools directly, Genie gracefully, UC tools through validated UC execution.
                if tc["name"] == "load_skill":
                    future = executor.submit(load_skill.invoke, tc["args"])
                    futures[future] = {"id": tc["id"], "name": tc["name"], "args": tc["args"]}
                elif tc["name"].startswith("$GENIE"):
                    future = executor.submit(
                        lambda: "Genie is not a direct tool. It will be consulted automatically "
                                "if the UC function results are insufficient. Continue with your "
                                "analysis using the UC function data you already have."
                    )
                    futures[future] = {"id": tc["id"], "name": tc["name"], "args": tc["args"]}
                else:
                    normalized_args, missing_required = _normalize_tool_args(tc["name"], tc.get("args"))
                    if missing_required:
                        tool_results.append({
                            "id": tc["id"],
                            "name": tc["name"],
                            "args": normalized_args,
                            "result": f"Error: Missing required tool parameter(s): {', '.join(missing_required)}",
                            "duration": 0.0,
                            "error": True,
                            "failure_type": "missing_required_parameter",
                        })
                        continue
                    future = executor.submit(_execute_uc_function_with_retries, tc["name"], normalized_args)
                    futures[future] = {"id": tc["id"], "name": tc["name"], "args": normalized_args}

            for future in as_completed(futures):
                tool_info = futures[future]
                elapsed_tool = time.time() - tool_start_times[tool_info["id"]]
                try:
                    result = future.result()
                    error = False
                    failure_type = None
                except Exception as e:
                    result = f"Error: {str(e)}"
                    error = True
                    failure_type = "transient_execution" if _is_transient_tool_error(e) else "execution_error"

                tool_results.append({
                    "id": tool_info["id"],
                    "name": tool_info["name"],
                    "args": tool_info["args"],
                    "result": result,
                    "duration": elapsed_tool,
                    "error": error,
                    "failure_type": failure_type,
                })

        elapsed = time.time() - start_time
        logger.info(f"Parallel tool execution: {len(tool_results)} tools in {elapsed:.2f}s")

        # MLflow tracing for parallel tool execution
        with mlflow.start_span(
            name=f"parallel_tools ({len(tool_results)} tools, {elapsed:.2f}s)",
            span_type=SpanType.TOOL
        ) as parent_span:
            parent_span.set_inputs({
                "num_tools": len(tool_results),
                "execution_mode": "parallel_mcp",
                "tool_names": [tr["name"] for tr in tool_results]
            })
            parent_span.set_attribute("wall_time_seconds", elapsed)

            for tr in tool_results:
                with mlflow.start_span(
                    name=f"{tr['name']} ({tr['duration']:.2f}s)",
                    span_type=SpanType.TOOL
                ) as span:
                    span.set_inputs({"tool_name": tr["name"], "args": tr["args"]})
                    span.set_outputs({
                        "execution_time_seconds": tr["duration"],
                        "result_preview": tr["result"][:500] if tr["result"] else "",
                        "error": tr["error"],
                        "failure_type": tr.get("failure_type"),
                    })

            parent_span.set_outputs({
                "completed": len(tool_results),
                "total_wall_time_seconds": elapsed,
                "failure_types": [tr.get("failure_type") for tr in tool_results if tr.get("failure_type")],
            })

        tool_messages = [
            ToolMessage(content=tr["result"], tool_call_id=tr["id"])
            for tr in tool_results
        ]

        # For deferred tool calls, return placeholder ToolMessages so every
        # tool_use block has a matching tool_result (required by the API).
        # The message tells the LLM to re-issue these calls now that it has
        # the skill instructions in context.
        for dtc in _deferred_calls:
            tool_messages.append(
                ToolMessage(
                    content=(
                        "This tool was not executed yet. You loaded a skill first. "
                        "Now that you have read the skill instructions, please call "
                        "this tool again with the appropriate parameters."
                    ),
                    tool_call_id=dtc["id"],
                )
            )

        return {"messages": tool_messages, "tool_calls_made": tool_results}

    def _summarize_tool_calls_for_eval(self, tool_calls_made: list[dict[str, Any]]) -> str:
        """Create a compact, judge-friendly summary of tool activity."""
        if not tool_calls_made:
            return "No tool calls were made."

        summaries = []
        for idx, tc in enumerate(tool_calls_made[:8], start=1):
            args = tc.get("args")
            if isinstance(args, dict):
                args_text = json.dumps(args, sort_keys=True)
            else:
                args_text = str(args)
            if len(args_text) > 300:
                args_text = args_text[:300] + "..."

            result_text = tc.get("result") or ""
            result_text = str(result_text)
            if len(result_text) > 600:
                result_text = result_text[:600] + "..."

            status = "error" if tc.get("error") else "ok"
            summaries.append(
                f"Tool {idx}: name={tc.get('name')} | status={status} | args={args_text} | output={result_text}"
            )

        if len(tool_calls_made) > 8:
            summaries.append(f"... {len(tool_calls_made) - 8} additional tool calls omitted")

        return "\n".join(summaries)

    def _parse_tool_payload(self, result: Any) -> dict[str, Any] | None:
        """Parse Databricks UC tool payloads without losing nested JSON evidence."""
        if isinstance(result, dict):
            return result
        if result is None:
            return None
        try:
            return json.loads(str(result))
        except Exception:
            return None

    def _extract_rows_for_evidence(self, payload: dict[str, Any] | None) -> list[dict[str, Any]]:
        if not isinstance(payload, dict):
            return []

        columns = payload.get("columns") or []
        rows = payload.get("rows") or []
        if not rows:
            return []

        if columns == ["output"] and rows and rows[0]:
            raw = rows[0][0]
            if raw in (None, "", "[]"):
                return []
            try:
                nested = json.loads(raw) if isinstance(raw, str) else raw
            except Exception:
                nested = None
            if isinstance(nested, list):
                return [item for item in nested if isinstance(item, dict)]
            if isinstance(nested, dict):
                return [nested]

        normalized = []
        for row in rows:
            if isinstance(row, dict):
                normalized.append(row)
            elif isinstance(row, list):
                normalized.append({str(col): val for col, val in zip(columns, row)})
        return normalized

    def _has_successful_tool_evidence(self, tool_calls_made: list[dict[str, Any]]) -> bool:
        for tc in tool_calls_made or []:
            if tc.get("error"):
                continue
            name = str(tc.get("name") or "")
            if name == "load_skill" or name.startswith("$GENIE"):
                continue
            payload = self._parse_tool_payload(tc.get("result"))
            if self._extract_rows_for_evidence(payload):
                return True
        return False

    def _looks_unavailable(self, text: str | None) -> bool:
        text_l = (text or "").lower()
        unavailable_markers = [
            "data provided does not include",
            "does not include information",
            "do not have information",
            "don't have information",
            "data is unavailable",
            "not available",
            "wasn't able to find",
            "couldn't fully answer",
            "cannot answer",
            "can't describe",
        ]
        return any(marker in text_l for marker in unavailable_markers)

    def _fmt_num(self, value: Any) -> str:
        try:
            number = float(value)
        except Exception:
            return str(value)
        if number.is_integer():
            return str(int(number))
        return f"{number:.2f}".rstrip("0").rstrip(".")

    def _arg_text(self, args: Any) -> str:
        if not isinstance(args, dict):
            return ""
        parts = []
        label_map = {
            "season_year": "season",
            "b_hand": "batter_hand",
            "p_hand": "pitcher_hand",
            "b": "balls",
            "s": "strikes",
            "p_on_1b": "runner_on_1b",
            "p_on_2b": "runner_on_2b",
            "p_on_3b": "runner_on_3b",
            "team": "team",
            "team_abbrev": "team",
        }
        for key, label in label_map.items():
            if key in args and args[key] not in (None, ""):
                parts.append(f"{label}={args[key]}")
        return ", ".join(parts)

    def _summarize_pitch_rows(self, rows: list[dict[str, Any]], limit: int = 6) -> tuple[str, int]:
        pitch_totals: dict[str, dict[str, Any]] = {}
        zone_totals: dict[str, float] = {}
        total = 0.0
        for row in rows:
            pitch_type = str(row.get("pitch_type") or row.get("pitch_name") or "unknown")
            pitch_name = row.get("pitch_name") or pitch_type
            try:
                count = float(row.get("pitch_count") or row.get("pitches") or row.get("count") or 0)
            except Exception:
                count = 0.0
            if count <= 0:
                count = 1.0 if pitch_type != "unknown" else 0.0
            entry = pitch_totals.setdefault(pitch_type, {"pitch_name": pitch_name, "pitch_count": 0.0})
            entry["pitch_count"] += count
            total += count
            zone = row.get("location_zone")
            if zone:
                zone_totals[str(zone)] = zone_totals.get(str(zone), 0.0) + count

        if not pitch_totals:
            return "No pitch mix rows were available.", 0

        ranked = sorted(pitch_totals.items(), key=lambda kv: kv[1]["pitch_count"], reverse=True)[:limit]
        pitch_lines = []
        for code, info in ranked:
            count = info["pitch_count"]
            pct = (count / total * 100.0) if total else 0.0
            pitch_lines.append(f"{info['pitch_name']} ({code}): {self._fmt_num(count)} pitches, {pct:.1f}%")

        zone_lines = []
        for zone, count in sorted(zone_totals.items(), key=lambda kv: kv[1], reverse=True)[:5]:
            pct = (count / total * 100.0) if total else 0.0
            zone_lines.append(f"{zone}: {self._fmt_num(count)} pitches, {pct:.1f}%")

        text = "Pitch mix: " + "; ".join(pitch_lines)
        if zone_lines:
            text += "\nLocation tendencies: " + "; ".join(zone_lines)
        return text, int(total) if total.is_integer() else int(round(total))

    def _format_tendency_evidence(self, tool_short: str, args: Any, rows: list[dict[str, Any]]) -> str:
        pitch_text, sample_size = self._summarize_pitch_rows(rows)
        filters = self._arg_text(args)
        sample_note = "Small sample; treat as directional." if sample_size and sample_size < 25 else "Use as primary tendency evidence."
        return (
            f"Tool: {tool_short}\n"
            f"Filters: {filters or 'not provided'}\n"
            f"Sample: {sample_size} pitches\n"
            f"{pitch_text}\n"
            f"Answer constraints: answer the exact count, handedness, season, and runner-state filters; {sample_note} Do not say data is unavailable."
        )

    def _format_matchup_evidence(self, tool_short: str, args: Any, rows: list[dict[str, Any]]) -> str:
        filters = self._arg_text(args)
        pitch_text, sample_size = self._summarize_pitch_rows(rows)
        outcome_keys = ["events", "description", "result", "launch_speed", "woba_value", "estimated_woba_using_speedangle"]
        outcome_examples = []
        for row in rows[:8]:
            parts = []
            for key in outcome_keys:
                if key in row and row[key] not in (None, ""):
                    parts.append(f"{key}={row[key]}")
            if parts:
                outcome_examples.append(", ".join(parts))
        sample_note = "Direct matchup sample is thin; use it as directional and lean on broader tendencies." if sample_size < 25 else "Direct matchup sample is usable for the recommendation."
        text = (
            f"Tool: {tool_short}\n"
            f"Filters: {filters or 'direct batter-pitcher matchup'}\n"
            f"Sample: {sample_size or len(rows)} rows/pitches\n"
            f"{pitch_text}\n"
        )
        if outcome_examples:
            text += "Outcome examples: " + " | ".join(outcome_examples[:4]) + "\n"
        text += f"Answer constraints: {sample_note} Separate observed matchup evidence from broader inference."
        return text

    def _format_arsenal_evidence(self, tool_short: str, args: Any, rows: list[dict[str, Any]]) -> str:
        filters = self._arg_text(args)
        pitches = []
        for row in rows:
            code = row.get("pitch_type") or row.get("pitch_code") or row.get("pitch")
            name = row.get("pitch_name") or row.get("name") or code
            if code or name:
                label = f"{name} ({code})" if code and name and str(code) != str(name) else str(name or code)
                if label not in pitches:
                    pitches.append(label)
        if not pitches:
            pitch_text, _ = self._summarize_pitch_rows(rows)
            pitches = [pitch_text]
        return (
            f"Tool: {tool_short}\n"
            f"Filters: {filters or 'pitcher arsenal'}\n"
            f"Pitch inventory: {', '.join(pitches[:12])}\n"
            "Answer constraints: this is arsenal inventory only. Do not infer usage rates, sequencing, or locations from arsenal rows alone."
        )

    def _format_roster_evidence(self, tool_short: str, args: Any, rows: list[dict[str, Any]]) -> str:
        filters = self._arg_text(args)
        players = []
        for row in rows[:20]:
            first = str(row.get("name_first") or row.get("first_name") or "").title()
            last = str(row.get("name_last") or row.get("last_name") or "").title()
            name = (first + " " + last).strip() or str(row.get("player_name") or row.get("name") or "")
            bats = row.get("bats") or row.get("stand") or row.get("batter_hand")
            extra = f" bats={bats}" if bats else ""
            if name:
                players.append(f"{name}{extra}")
        return (
            f"Tool: {tool_short}\n"
            f"Filters: {filters or 'team roster'}\n"
            f"Returned players: {len(rows)}\n"
            f"Players: {', '.join(players[:15])}\n"
            "Answer constraints: list only players returned by the tool. Do not invent roster members."
        )

    def _format_recommendation_evidence(self, tool_short: str, args: Any, rows: list[dict[str, Any]]) -> str:
        filters = self._arg_text(args)
        examples = []
        for row in rows[:10]:
            parts = []
            for key, value in row.items():
                if value not in (None, "") and len(parts) < 8:
                    parts.append(f"{key}={value}")
            if parts:
                examples.append(", ".join(parts))
        return (
            f"Tool: {tool_short}\n"
            f"Filters: {filters or 'team matchup recommendation'}\n"
            f"Returned recommendations: {len(rows)}\n"
            f"Recommendation rows: {' | '.join(examples[:6])}\n"
            "Answer constraints: rank or summarize recommendations using only returned rows and explain the baseball reason if present in the evidence."
        )

    def _format_lookup_evidence(self, tool_short: str, args: Any, rows: list[dict[str, Any]]) -> str:
        matches = []
        for row in rows[:8]:
            first = str(row.get("name_first") or "").title()
            last = str(row.get("name_last") or "").title()
            player_id = row.get("player_id")
            player_type = row.get("player_type") or "player"
            throws = row.get("throws")
            bats = row.get("bats")
            hand = ", ".join([x for x in [f"throws={throws}" if throws else "", f"bats={bats}" if bats else ""] if x])
            matches.append(f"{first} {last} ({player_type}, id={player_id}{', ' + hand if hand else ''})")
        return (
            f"Tool: {tool_short}\n"
            f"Lookup matches: {', '.join(matches)}\n"
            "Answer constraints: use lookup rows only to resolve identity and handedness. This is supporting evidence, not the substantive answer."
        )

    def _format_generic_evidence(self, tool_short: str, args: Any, rows: list[dict[str, Any]]) -> str:
        filters = self._arg_text(args)
        examples = []
        for row in rows[:5]:
            parts = []
            for key, value in list(row.items())[:8]:
                parts.append(f"{key}={value}")
            examples.append(", ".join(parts))
        return (
            f"Tool: {tool_short}\n"
            f"Filters: {filters or 'not provided'}\n"
            f"Returned rows: {len(rows)}\n"
            f"Examples: {' | '.join(examples)}\n"
            "Answer constraints: answer only from these returned rows."
        )

    def _format_tool_result_for_draft(self, tool_name: str, result: Any, args: Any = None) -> str:
        """Create deterministic, tool-aware evidence briefs from UC tool payloads."""
        payload = self._parse_tool_payload(result)
        if not isinstance(payload, dict):
            return str(result or "")[:1200]

        rows = payload.get("rows") or []
        evidence_rows = self._extract_rows_for_evidence(payload)
        if not rows or not evidence_rows:
            return f"{tool_name} returned no rows."

        tool_short = tool_name.split("__")[-1]
        if "get_pitcher_tendency_by_count" in tool_name or "get_pitcher_tendency_with_runners" in tool_name:
            return self._format_tendency_evidence(tool_short, args, evidence_rows)
        if "get_batter_pitcher_matchup" in tool_name:
            return self._format_matchup_evidence(tool_short, args, evidence_rows)
        if "pitcher_arsenal_lookup" in tool_name:
            return self._format_arsenal_evidence(tool_short, args, evidence_rows)
        if "get_team_batters" in tool_name:
            return self._format_roster_evidence(tool_short, args, evidence_rows)
        if "recommend_batter_matchups_by_team" in tool_name:
            return self._format_recommendation_evidence(tool_short, args, evidence_rows)
        if "lookup_player_by_name" in tool_name:
            return self._format_lookup_evidence(tool_short, args, evidence_rows)
        return self._format_generic_evidence(tool_short, args, evidence_rows)

    def _draft_uc_response_from_tools(self, original_query: str, tool_calls_made: list[dict[str, Any]]) -> str:
        """Draft a direct evidence brief when the model ignored usable evidence."""
        successful_calls = [tc for tc in (tool_calls_made or []) if not tc.get("error")]
        if not successful_calls:
            return ""

        evidence_parts = []
        supporting_parts = []
        for tc in successful_calls[:8]:
            formatted_result = self._format_tool_result_for_draft(
                tc.get("name", "tool"),
                tc.get("result"),
                tc.get("args"),
            )
            if not formatted_result or "returned no rows" in formatted_result:
                continue
            if "lookup_player_by_name" in str(tc.get("name", "")):
                supporting_parts.append(formatted_result)
            else:
                evidence_parts.append(formatted_result)

        if not evidence_parts and not supporting_parts:
            return ""

        sections = [
            "EVIDENCE BRIEF",
            f"Question: {original_query}",
        ]
        if supporting_parts:
            sections.append("Supporting identity evidence:\n" + "\n\n".join(supporting_parts[:3]))
        if evidence_parts:
            sections.append("Substantive tool evidence:\n" + "\n\n".join(evidence_parts[:6]))
        sections.append(
            "Global answer constraints: answer the user's exact question from the evidence above; "
            "include sample-size caveats when noted; do not claim data is unavailable when substantive tool evidence is present."
        )
        return "\n\n".join(sections)

    def _synthesis_model_invoke(self, prompt: str) -> str:
        model = getattr(self, "synthesis_model", None) or self.model
        try:
            result = model.invoke([HumanMessage(content=prompt)])
        except Exception as e:
            logger.warning(f"Final synthesis model failed, falling back to runtime model: {e}")
            result = self.model.invoke([HumanMessage(content=prompt)])
        return _strip_reasoning_artifacts(result.content)

    def _evaluate_sufficiency(self, state: AgentState) -> dict:
        """Evaluate if UC response is sufficient, partial, or not answered."""
        original_query = state.get("original_query", "")
        uc_response = state.get("uc_response", "")
        tool_calls_made = state.get("tool_calls_made", [])
        tool_summary = self._summarize_tool_calls_for_eval(tool_calls_made)
        uc_response_source = "agent_response"

        if tool_calls_made and (not (uc_response or "").strip() or self._looks_unavailable(uc_response)):
            drafted_response = self._draft_uc_response_from_tools(original_query, tool_calls_made)
            if drafted_response.strip():
                uc_response = drafted_response
                uc_response_source = "tool_draft"

        eval_prompt = f"""You are evaluating whether a response adequately answers a user's question.
The question may have MULTIPLE parts. Evaluate EACH part separately.

Original Question: {original_query}

Draft Response from UC Functions Agent:
{uc_response or '[EMPTY RESPONSE]'}

Observed Tool Calls and Outputs:
{tool_summary}

Evaluation rules:
- Judge the drafted response, not whether tools were merely called.
- Use the tool outputs as supporting evidence to decide whether the drafted response answered the question.
- FULLY_ANSWERED means the response addresses every material part of the question, even if the answer is that no relevant records exist.
- PARTIALLY_ANSWERED means the response answers only some parts, omits a requested recommendation, or ignores relevant evidence.
- NOT_ANSWERED means the response is empty, only reports an execution failure, or does not materially address the user's request.
- If the tool outputs contain relevant evidence that the response failed to use, prefer PARTIALLY_ANSWERED over FULLY_ANSWERED.

Respond in this EXACT format:
STATUS: [FULLY_ANSWERED or PARTIALLY_ANSWERED or NOT_ANSWERED]
ANSWERED_PARTS: [Brief summary of what was answered, or "None"]
UNANSWERED_PARTS: [Specific questions that still need answers, or "None"]
RATIONALE: [One brief sentence]

Your evaluation:"""

        eval_result = self.model.invoke([HumanMessage(content=eval_prompt)])
        eval_text = _strip_reasoning_artifacts(eval_result.content)
        eval_content = eval_text.upper()

        is_sufficient = "FULLY_ANSWERED" in eval_content
        is_partial = "PARTIALLY_ANSWERED" in eval_content

        answered_parts = None
        unanswered_parts = None
        rationale = None

        try:
            lines = eval_text.split('\n')
            for line in lines:
                upper_line = line.upper()
                if upper_line.startswith("ANSWERED_PARTS:"):
                    answered_parts = line.split(":", 1)[1].strip()
                    if answered_parts.upper() == "NONE":
                        answered_parts = None
                elif upper_line.startswith("UNANSWERED_PARTS:"):
                    unanswered_parts = line.split(":", 1)[1].strip()
                    if unanswered_parts.upper() == "NONE":
                        unanswered_parts = None
                elif upper_line.startswith("RATIONALE:"):
                    rationale = line.split(":", 1)[1].strip()
        except Exception as e:
            logger.warning(f"Failed to parse sufficiency evaluation: {e}")

        if is_partial and unanswered_parts:
            status = "PARTIALLY_ANSWERED"
            result = {
                "uc_sufficient": False,
                "uc_answered_parts": answered_parts,
                "unanswered_parts": unanswered_parts,
                "uc_response": uc_response,
            }
        elif is_sufficient:
            status = "FULLY_ANSWERED"
            result = {"uc_sufficient": True, "uc_answered_parts": answered_parts, "unanswered_parts": None, "uc_response": uc_response}
        else:
            status = "NOT_ANSWERED"
            result = {"uc_sufficient": False, "uc_answered_parts": None, "unanswered_parts": original_query, "uc_response": uc_response}

        with mlflow.start_span(name=f"sufficiency_eval ({status})", span_type=SpanType.LLM) as span:
            span.set_inputs({
                "original_query": original_query,
                "uc_response": uc_response,
                "uc_response_source": uc_response_source,
                "tool_calls_summary": tool_summary,
                "num_tool_calls": len(tool_calls_made),
                "eval_prompt": eval_prompt,
            })
            span.set_outputs({
                "status": status,
                "answered_parts": answered_parts,
                "unanswered_parts": result["unanswered_parts"],
                "rationale": rationale,
                "will_fallback_to_genie": not result["uc_sufficient"],
                "raw_eval_text": eval_text,
            })

        return result

    def _genie_fallback_node(self, state: AgentState) -> dict:
        """Fall back to Genie for unanswered parts.

        Before executing the Genie query, dynamically loads any skills whose
        description mentions Genie so the agent has relevant guidance.
        """
        original_query = state.get("original_query", "")
        unanswered_parts = state.get("unanswered_parts", "")

        if not self.genie_agent:
            return {
                "genie_response": "",
                "genie_error": "Genie is disabled because no Genie Space ID is configured.",
            }

        # Dynamically load Genie-related skills before the fallback query
        if SKILL_METADATA_BLOCK and _skill_rows:
            genie_skills = []
            try:
                for skill in _skill_rows:
                    name = skill["name"]
                    description = skill["description"]
                    if "genie" in name.lower() or "genie" in description.lower():
                        genie_skills.append({
                            "name": name,
                            "content": _query_skill_content(skill["filename"]),
                        })

                if genie_skills:
                    with mlflow.start_span(name="load_genie_skills", span_type=SpanType.RETRIEVER) as span:
                        span.set_inputs({"count": len(genie_skills), "names": [s["name"] for s in genie_skills]})
                        for gs in genie_skills:
                            logger.info(f"Loaded Genie skill: {gs['name']}")
                        span.set_outputs({"loaded": [s["name"] for s in genie_skills]})
            except Exception as e:
                genie_skills = []
                logger.warning(f"Could not load Genie skills: {e}")
        else:
            genie_skills = []

        if unanswered_parts and unanswered_parts != original_query:
            genie_query = f"""I need help answering a specific part of a question about baseball data.
The user's full question was: {original_query}

I was able to answer part of it, but I need your help with:
{unanswered_parts}

Please focus on answering ONLY the part I couldn't answer."""
        else:
            genie_query = f"""I need help answering this question about baseball data.
Question: {original_query}
Please use your data access capabilities to answer this question."""

        # If Genie skills were loaded, prepend their guidance to the query
        if genie_skills:
            skill_guidance = "\n\n".join(
                f"[Skill: {gs['name']}]\n{gs['content']}" for gs in genie_skills
            )
            genie_query = (
                f"The following skill guidance applies to this query:\n\n"
                f"{skill_guidance}\n\n---\n\n{genie_query}"
            )

        genie_messages = [HumanMessage(content=genie_query)]

        genie_response = ""
        genie_error = None
        start_time = time.time()

        for attempt in range(self.genie_config.max_retries + 1):
            try:
                logger.info(f"Genie fallback attempt {attempt + 1}")
                result = self.genie_agent.invoke({"messages": genie_messages})

                for msg in reversed(result.get("messages", [])):
                    if hasattr(msg, "content") and msg.content:
                        genie_response = _strip_reasoning_artifacts(msg.content)
                        break

                if genie_response and genie_response.strip().upper() not in {"EMPTY", "NO DATA", "NO RESULTS"}:
                    break
                genie_response = ""

            except Exception as e:
                genie_error = f"Genie error: {str(e)}"
                logger.warning(f"Genie error on attempt {attempt + 1}: {e}")
                if attempt < self.genie_config.max_retries:
                    time.sleep(2 ** attempt)

        elapsed = time.time() - start_time

        with mlflow.start_span(name=f"genie_fallback ({elapsed:.2f}s)", span_type=SpanType.CHAIN) as span:
            span.set_inputs({"genie_space_id": self.genie_config.space_id, "original_query": original_query})
            span.set_outputs({"success": bool(genie_response), "duration_seconds": elapsed})

        return {"genie_response": genie_response, "genie_error": genie_error}

    def _build_fallback_action_plan(self, state: AgentState) -> str:
        """Return a useful terminal response when exact retrieval fails or is empty."""
        original_query = state.get("original_query", "")
        uc_response = state.get("uc_response", "")
        tool_calls_made = state.get("tool_calls_made", [])
        genie_error = state.get("genie_error")

        checked = []
        failures = []
        empty_results = []
        for tc in tool_calls_made[:8]:
            name = str(tc.get("name", "tool")).split("__")[-1]
            result_text = str(tc.get("result") or "")
            if tc.get("error"):
                failures.append(name)
            elif "rows" in result_text or result_text.strip():
                checked.append(name)
                if "\"rows\": []" in result_text or "returned no rows" in result_text.lower():
                    empty_results.append(name)

        checked_text = ", ".join(dict.fromkeys(checked)) or "the available baseball data tools"
        failure_text = ", ".join(dict.fromkeys(failures)) or None
        empty_text = ", ".join(dict.fromkeys(empty_results)) or None

        data_notes = [f"Checked {checked_text}."]
        if failure_text:
            data_notes.append(f"Some retrieval paths failed or timed out: {failure_text}.")
        if empty_text:
            data_notes.append(f"Some successful lookups returned no rows: {empty_text}.")
        if genie_error:
            data_notes.append(f"Genie fallback was unavailable: {genie_error}.")
        if uc_response:
            data_notes.append(f"Partial evidence: {uc_response[:500]}")

        return f"""# At-Bat Assistant Assessment
## Data collected
- {' '.join(data_notes)}

## Pitcher Approach
- The exact matchup slice was not fully available, so treat this as a preparation framework rather than a definitive scouting report. Anchor the plan on handedness, count leverage, pitch mix, location patterns, and runner state. Look for fastball counts early, protect against the primary secondary pitch with two strikes, and avoid expanding to chase zones until the pitcher proves he can land those pitches for strikes.

## Recommendation
- Use a selective, count-aware approach: hunt a pitch in one zone before two strikes, shrink the zone with runners on base, and force the pitcher to execute secondary stuff in the strike zone. If you need a precise recommendation, rerun with the pitcher, batter, season, count, handedness, and base state explicitly specified.
"""

    def _synthesize_response(self, state: AgentState) -> dict:
        """Synthesize final response from UC and Genie outputs."""
        uc_response = state.get("uc_response", "")
        genie_response = state.get("genie_response", "")
        genie_error = state.get("genie_error")
        uc_sufficient = state.get("uc_sufficient", False)
        uc_answered_parts = state.get("uc_answered_parts")
        unanswered_parts = state.get("unanswered_parts")
        original_query = state.get("original_query", "")
        tool_calls_made = state.get("tool_calls_made", [])
        tool_evidence_summary = self._draft_uc_response_from_tools(original_query, tool_calls_made) or self._summarize_tool_calls_for_eval(tool_calls_made)

        if uc_sufficient:
            if self._looks_unavailable(uc_response) and self._has_successful_tool_evidence(tool_calls_made):
                uc_response = self._draft_uc_response_from_tools(original_query, tool_calls_made) or uc_response
            final_response = uc_response
        elif genie_response:
            if uc_answered_parts and unanswered_parts != original_query:
                synthesis_prompt = f"""Combine two partial responses into a unified answer.

CURRENT QUESTION (answer ONLY this): {original_query}

PART 1 - From UC Functions:
{uc_response}

RAW TOOL EVIDENCE FROM UC FUNCTIONS:
{tool_evidence_summary}

PART 2 - From Genie:
{genie_response}

IMPORTANT: Only include information that DIRECTLY answers the current question above.
If the UC draft says data is unavailable but RAW TOOL EVIDENCE contains successful non-empty rows, ignore the unavailable claim and synthesize the raw evidence.
Do NOT include information from prior conversation turns unless explicitly referenced.
Create a UNIFIED response that presents all relevant information clearly without mentioning different systems."""
            else:
                synthesis_prompt = f"""Synthesize a response based on available information.

CURRENT QUESTION (answer ONLY this): {original_query}

Available Information:
- UC Functions: {uc_response}
- Raw UC Tool Evidence: {tool_evidence_summary}
- Genie: {genie_response}

IMPORTANT: Only include information that DIRECTLY answers the current question.
If the UC draft says data is unavailable but Raw UC Tool Evidence contains successful non-empty rows, ignore the unavailable claim and synthesize the raw evidence.
Provide a clear, focused response."""

            final_response = self._synthesis_model_invoke(synthesis_prompt)

        elif genie_error:
            if uc_response and uc_answered_parts:
                final_response = f"""{uc_response}

---
{self._build_fallback_action_plan(state)}"""
            else:
                final_response = self._build_fallback_action_plan(state)
        else:
            if tool_calls_made:
                synthesis_prompt = f"""Synthesize a response based on available UC tool evidence.

CURRENT QUESTION (answer ONLY this): {original_query}

Draft response from UC Functions Agent:
{uc_response or '[EMPTY RESPONSE]'}

RAW TOOL EVIDENCE FROM UC FUNCTIONS:
{tool_evidence_summary}

IMPORTANT: If the draft says data is unavailable but RAW TOOL EVIDENCE contains successful non-empty rows, ignore the unavailable claim and answer from the raw evidence.
If tool evidence is empty or only errors, state the specific limitation. Otherwise summarize pitch mix, locations, and an actionable hitter recommendation when relevant."""
                final_response = self._synthesis_model_invoke(synthesis_prompt)
            else:
                final_response = uc_response or self._build_fallback_action_plan(state)

        inventory_query = bool(re.search(r"\b(what|which|list)\b.*\b(pitch types|pitches|arsenal)\b", original_query.lower()))
        if inventory_query:
            pitch_codes = []
            pitcher_name = "the pitcher"
            for tr in tool_calls_made:
                name = tr.get("name", "")
                try:
                    payload = json.loads(tr.get("result") or "{}")
                except Exception:
                    payload = {}
                if name.endswith("lookup_player_by_name"):
                    rows = payload.get("rows") or []
                    if rows and len(rows[0]) >= 3:
                        pitcher_name = f"{str(rows[0][1]).title()} {str(rows[0][2]).title()}"
                if name.endswith("pitcher_arsenal_lookup"):
                    for row in payload.get("rows") or []:
                        if len(row) >= 2 and row[1] and row[1] not in pitch_codes:
                            pitch_codes.append(row[1])
            if pitch_codes:
                pitch_name_map = {
                    "FF": "4-Seam Fastball", "FC": "Cutter", "CH": "Changeup",
                    "SI": "Sinker", "ST": "Sweeper", "SL": "Slider", "CU": "Curveball",
                    "KC": "Knuckle Curve", "FS": "Splitter", "PO": "Pitchout",
                }
                pitch_lines = [f"- {code}: {pitch_name_map.get(code, 'Unknown pitch type')}" for code in pitch_codes]
                final_response = (
                    "# At-Bat Assistant Assessment\n\n"
                    "## Data collected\n"
                    f"- {pitcher_name}'s arsenal lookup returned {len(pitch_codes)} pitch types.\n\n"
                    "## Pitcher Approach\n"
                    + "\n".join(pitch_lines)
                    + "\n\nNo usage rates, sequencing, speeds, movement, or count-specific tendencies were requested or retrieved, so I am not inferring them from this arsenal-only lookup.\n\n"
                    "## Recommendation\n"
                    "Use this as pitch-type inventory only. For a hitter game plan, ask for a specific batter, count, handedness, or runner state so the tendency tools can ground the recommendation."
                )

        final_response = _strip_reasoning_artifacts(final_response)

        return {"final_response": final_response, "messages": [AIMessage(content=final_response)]}

    def _create_graph(self, checkpointer=None):
        """Create the LangGraph workflow with UC-first, Genie-fallback pattern."""

        def should_continue_tools(state: AgentState):
            messages = state["messages"]
            last_message = messages[-1] if messages else None
            if isinstance(last_message, AIMessage) and last_message.tool_calls:
                return "tools"
            return "evaluate"

        def route_after_evaluation(state: AgentState) -> Literal["genie_fallback", "synthesize"]:
            if state.get("uc_sufficient", False):
                return "synthesize"
            if self._has_successful_tool_evidence(state.get("tool_calls_made", [])):
                return "synthesize"
            if not self.genie_agent:
                return "synthesize"
            return "genie_fallback"

        # Preprocessor with system prompt (append skill metadata if available)
        if self.system_prompt:
            base_prompt_text = self.system_prompt.format() if hasattr(self.system_prompt, 'format') else str(self.system_prompt)
            prompt_text = (base_prompt_text + SKILL_METADATA_BLOCK) if SKILL_METADATA_BLOCK else base_prompt_text
            preprocessor = RunnableLambda(
                lambda state: [{"role": "system", "content": prompt_text}] + list(state["messages"])
            )
        else:
            preprocessor = RunnableLambda(lambda state: list(state["messages"]))

        model_runnable = preprocessor | self.model_with_tools

        def call_model(state: AgentState, config: RunnableConfig):
            response = model_runnable.invoke(state, config)
            if isinstance(response, AIMessage) and response.content:
                response.content = _strip_reasoning_artifacts(response.content)
            original_query = state.get("original_query", "")
            if not original_query:
                for msg in reversed(state["messages"]):
                    if isinstance(msg, HumanMessage):
                        original_query = msg.content
                        break
            return {"messages": [response], "original_query": original_query}

        def extract_uc_response(state: AgentState):
            messages = state["messages"]
            uc_response = ""
            for msg in reversed(messages):
                if isinstance(msg, AIMessage) and msg.content and not msg.tool_calls:
                    uc_response = _strip_reasoning_artifacts(msg.content)
                    break
            return {"uc_response": uc_response}

        workflow = StateGraph(AgentState)

        # Add nodes
        workflow.add_node("agent", RunnableLambda(call_model))
        workflow.add_node("tools", self._parallel_tool_node)
        workflow.add_node("extract_response", extract_uc_response)
        workflow.add_node("evaluate", self._evaluate_sufficiency)
        workflow.add_node("genie_fallback", self._genie_fallback_node)
        workflow.add_node("synthesize", self._synthesize_response)

        # Add edges
        workflow.set_entry_point("agent")
        workflow.add_conditional_edges(
            "agent",
            should_continue_tools,
            {"tools": "tools", "evaluate": "extract_response"}
        )
        workflow.add_edge("tools", "agent")
        workflow.add_edge("extract_response", "evaluate")
        workflow.add_conditional_edges(
            "evaluate",
            route_after_evaluation,
            {"genie_fallback": "genie_fallback", "synthesize": "synthesize"}
        )
        workflow.add_edge("genie_fallback", "synthesize")
        workflow.add_edge("synthesize", END)

        return workflow.compile(checkpointer=checkpointer)

    def _get_or_create_thread_id(self, request: ResponsesAgentRequest) -> str:
        ci = dict(request.custom_inputs or {})
        if "thread_id" in ci:
            return ci["thread_id"]
        if request.context and getattr(request.context, "conversation_id", None):
            return request.context.conversation_id
        return str(uuid.uuid4())

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        thread_id = self._get_or_create_thread_id(request)
        ci = dict(request.custom_inputs or {})
        ci["thread_id"] = thread_id
        request.custom_inputs = ci

        outputs = [
            event.item
            for event in self.predict_stream(request)
            if event.type == "response.output_item.done"
        ]
        return ResponsesAgentResponse(output=outputs, custom_outputs={"thread_id": thread_id})

    def predict_stream(
        self,
        request: ResponsesAgentRequest,
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        """Streaming prediction with PostgreSQL checkpointing and MLflow tracing."""
        thread_id = self._get_or_create_thread_id(request)

        ci = dict(request.custom_inputs or {})
        ci["thread_id"] = thread_id
        request.custom_inputs = ci

        if os.getenv("ATBAT_MLFLOW_LOG_MODEL_DRY_RUN", "false").lower() == "true":
            yield ResponsesAgentStreamEvent(
                type="response.output_item.done",
                item=self.create_text_output_item(
                    text="Model logging validation response. Live model calls are skipped during packaging.",
                    id=str(uuid.uuid4()),
                ),
                custom_outputs={"thread_id": thread_id},
            )
            return

        cc_msgs = self.prep_msgs_for_cc_llm([i.model_dump() for i in request.input])
        checkpoint_config = {"configurable": {"thread_id": thread_id}}

        # Convert to LangChain messages
        lc_messages = []
        for msg in cc_msgs:
            role = msg.get("role", "user")
            content = msg.get("content", "")
            if role == "user":
                lc_messages.append(HumanMessage(content=content))
            elif role == "assistant":
                lc_messages.append(AIMessage(content=content))

        # MLflow tracing for Lakebase session
        with mlflow.start_span(name="lakebase_session", span_type=SpanType.RETRIEVER) as db_span:
            db_span.set_inputs({
                "thread_id": thread_id,
                "lakebase_instance": self.lakebase_config.get("instance_name", "unknown"),
                "memory_enabled": bool(self._connection_pool)
            })

            with self.get_connection() as conn:
                checkpointer = PostgresSaver(conn) if conn else None
                graph = self._create_graph(checkpointer=checkpointer)

                node_count = 0
                final_response = None
                emitted_tool_calls = set()
                tracked_unanswered_parts = ""

                for event in graph.stream(
                    {"messages": lc_messages},
                    checkpoint_config,
                    stream_mode=["updates", "messages"]
                ):
                    if event[0] == "updates":
                        node_count += 1
                        node_name = list(event[1].keys())[0] if event[1] else ""
                        node_data = event[1].get(node_name, {})

                        # Emit function_call events when tools node executes
                        if node_name == "tools" and "tool_calls_made" in node_data:
                            for tc in node_data["tool_calls_made"]:
                                if tc["id"] not in emitted_tool_calls:
                                    emitted_tool_calls.add(tc["id"])
                                    yield ResponsesAgentStreamEvent(
                                        type="response.output_item.done",
                                        item=self.create_function_call_item(
                                            id=str(uuid.uuid4()),
                                            call_id=tc["id"],
                                            name=tc["name"],
                                            arguments=json.dumps(tc["args"]) if isinstance(tc["args"], dict) else tc["args"],
                                        ),
                                        custom_outputs={"thread_id": thread_id}
                                    )
                                    yield ResponsesAgentStreamEvent(
                                        type="response.output_item.done",
                                        item=self.create_function_call_output_item(
                                            call_id=tc["id"],
                                            output=tc["result"][:2000] if tc["result"] else "",
                                        ),
                                        custom_outputs={"thread_id": thread_id}
                                    )

                        if node_name == "evaluate" and "unanswered_parts" in node_data:
                            tracked_unanswered_parts = node_data.get("unanswered_parts", "")

                        # Emit Genie fallback event
                        if node_name == "genie_fallback":
                            genie_call_id = f"genie-{uuid.uuid4()}"
                            genie_response = node_data.get("genie_response", "")
                            query_display = tracked_unanswered_parts if tracked_unanswered_parts else "Genie fallback query"
                            yield ResponsesAgentStreamEvent(
                                type="response.output_item.done",
                                item=self.create_function_call_item(
                                    id=str(uuid.uuid4()),
                                    call_id=genie_call_id,
                                    name="genie_space_query",
                                    arguments=json.dumps({"query": query_display[:200]}),
                                ),
                                custom_outputs={"thread_id": thread_id}
                            )
                            yield ResponsesAgentStreamEvent(
                                type="response.output_item.done",
                                item=self.create_function_call_output_item(
                                    call_id=genie_call_id,
                                    output=genie_response[:500] if genie_response else "(queried Genie space)",
                                ),
                                custom_outputs={"thread_id": thread_id}
                            )

                        # Track final response from synthesize node
                        if node_name == "synthesize" and "final_response" in node_data:
                            final_response = node_data["final_response"]
                        elif node_name == "synthesize" and "messages" in node_data:
                            for msg in node_data["messages"]:
                                if isinstance(msg, AIMessage) and msg.content:
                                    final_response = msg.content

                # Emit final synthesized response
                if final_response:
                    yield ResponsesAgentStreamEvent(
                        type="response.output_item.done",
                        item=self.create_text_output_item(
                            text=final_response,
                            id=str(uuid.uuid4()),
                        ),
                        custom_outputs={"thread_id": thread_id}
                    )

                db_span.set_outputs({
                    "status": "completed",
                    "thread_id": thread_id,
                    "nodes_executed": node_count,
                    "tool_calls_emitted": len(emitted_tool_calls)
                })


########################################
# INSTANTIATE AGENT
########################################

AGENT = CombinedLangGraphAgent(LAKEBASE_CONFIG, GENIE_CONFIG)
mlflow.models.set_model(AGENT)


In [ ]:
dbutils.library.restartPython()

## Test the Agent

### Expected Behavior
1. **UC Functions First**: The agent always tries to answer using Unity Catalog functions first
2. **Sufficiency Check**: LLM evaluates if the UC response is FULLY, PARTIALLY, or NOT answered
3. **Partial Answer Detection**: If UC answered some parts but not others, only unanswered parts go to Genie
4. **Genie Fallback**: For unanswered parts, Genie is automatically invoked
5. **Response Synthesis**: The final response intelligently combines both sources

In [ ]:
# Reload config after Python restart
import json
import os
from pathlib import Path
import mlflow

CONFIG = json.loads(Path("config/atbat_assistant.json").read_text())

# Re-extract all configuration variables (needed for logging/deployment cells)
PROMPT_NAME = CONFIG["prompt_registry"]["prompt_name"]
LLM_ENDPOINT_NAME = CONFIG["llm"]["endpoint_name"]
FINAL_SYNTHESIS_ENDPOINT_NAME = os.getenv(
    "ATBAT_FINAL_SYNTHESIS_ENDPOINT",
    CONFIG["llm"].get("final_synthesis_endpoint_name", "gpt-5-4-external"),
)
FINAL_SYNTHESIS_ENDPOINT_NAME = str(FINAL_SYNTHESIS_ENDPOINT_NAME or "").replace("databricks:/", "")
UC_MODEL_NAME = CONFIG["model"]["uc_model_name"]
UC_TOOL_NAMES = CONFIG["tools"]["uc_tool_names"]
DISABLE_VECTOR_TOOLS = os.getenv("DISABLE_VECTOR_TOOLS", "true").lower() in {"1", "true", "yes"}
if DISABLE_VECTOR_TOOLS:
    UC_TOOL_NAMES = [
        tool_name for tool_name in UC_TOOL_NAMES
        if "embedding" not in tool_name.lower() and "vector" not in tool_name.lower()
    ]
CATALOG = CONFIG["workspace"]["catalog"]
SCHEMA = CONFIG["workspace"]["schema"]
GENIE_SPACE_ID = CONFIG["genie"]["space_id"]
GENIE_NAME = CONFIG["genie"]["name"]
LAKEBASE_INSTANCE = CONFIG["lakebase"]["instance_name"]

# Set MLflow experiment
EXPERIMENT_ID = CONFIG["mlflow"]["experiment_id"]
mlflow.set_experiment(experiment_id=EXPERIMENT_ID)


In [ ]:
import os
from agent_with_skills import AGENT

RUN_LIVE_AGENT_TESTS = os.getenv("RUN_LIVE_AGENT_TESTS", "false").lower() == "true"

# Test 1: UC Functions Only
if RUN_LIVE_AGENT_TESTS:
    print("=== Test 1: UC Functions Only ===")
    result = AGENT.predict({"input": [{"role": "user", "content": "How will Kyle Freeland pitch to Freddie Freeman?"}]})
    print(f"Thread ID: {result.custom_outputs.get('thread_id')}")
    print(f"Response preview: {result.output[-1].content[0]['text'][:500]}...")
else:
    print("Skipping live agent tests. Set RUN_LIVE_AGENT_TESTS=true to enable them.")

In [ ]:

# Test 2: UC Functions AND Genie
if RUN_LIVE_AGENT_TESTS:
    print("=== Test 2: UC Functions + Genie Fallback ===")
    result = AGENT.predict({"input": [{"role": "user", "content": "How will Blake Snell pitch to Mookie Betts? What was the full pitch distribution for the Rockies in 2025 including the average spin rate and velocity by pitch?"}]})
    print(f"Thread ID: {result.custom_outputs.get('thread_id')}")
    print(f"Response preview: {result.output[-1].content[0]['text'][:500]}...")
else:
    print("Skipping live Genie fallback test.")

In [ ]:
# Test 3: Streaming
if RUN_LIVE_AGENT_TESTS:
    print("=== Test 3: Streaming ===")
    for chunk in AGENT.predict_stream(
        {"input": [{"role": "user", "content": "How does Yu Darvish pitch to lefties with runners on 2nd?"}]}
    ):
        print(chunk.model_dump(exclude_none=True))
else:
    print("Skipping live streaming test.")

In [ ]:
# Test 4: Memory test (uses same thread_id)
if RUN_LIVE_AGENT_TESTS:
    print("=== Test 4: Memory Test ===")
    thread_id = "atbat-memory-test-001"

    result1 = AGENT.predict({
        "input": [{"role": "user", "content": "The demo memory phrase is inside fastball"}],
        "custom_inputs": {"thread_id": thread_id}
    })

    result2 = AGENT.predict({
        "input": [{"role": "user", "content": "What demo memory phrase did I provide?"}],
        "custom_inputs": {"thread_id": thread_id}
    })
    print(f"Memory response: {result2.output[-1].content[0]['text'][:300]}")
else:
    print("Skipping live memory test. Lakebase memory is disabled in Free Edition.")

## Log and Deploy the Agent

In [ ]:
import os

# Determine Databricks resources for automatic auth passthrough at deployment time
# Docs: https://docs.databricks.com/aws/en/generative-ai/agent-framework/agent-authentication
#
# IMPORTANT: Declare only resources that are configured and available. Free Edition
# workspaces can block Lakebase or Apps resource creation.
from mlflow.models.resources import (
    DatabricksFunction,
    DatabricksGenieSpace,
    DatabricksServingEndpoint,
    DatabricksSQLWarehouse,
    DatabricksTable,
)
from pkg_resources import get_distribution


def _is_configured(value):
    return bool(value) and not str(value).startswith("PLACEHOLDER_")


warehouse_id = CONFIG["genie"].get("warehouse_id", "")
genie_space_id = CONFIG["genie"].get("space_id", "")
uc_schema = f"{CATALOG}.{SCHEMA}"

resources = [DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT_NAME)]
final_synthesis_endpoint_name = FINAL_SYNTHESIS_ENDPOINT_NAME
if final_synthesis_endpoint_name and final_synthesis_endpoint_name != LLM_ENDPOINT_NAME:
    resources.append(DatabricksServingEndpoint(endpoint_name=final_synthesis_endpoint_name))

# Add UC functions from config.
for tool_name in UC_TOOL_NAMES:
    resources.append(DatabricksFunction(function_name=tool_name))

# Genie space + SQL warehouse are required only when Genie is configured.
if _is_configured(warehouse_id):
    resources.append(DatabricksSQLWarehouse(warehouse_id=warehouse_id))
else:
    print("Skipping SQL warehouse resource because no warehouse is configured.")

if _is_configured(genie_space_id):
    resources.append(DatabricksGenieSpace(genie_space_id=genie_space_id))
    for table in CONFIG["genie"].get("tables", []):
        resources.append(DatabricksTable(table_name=f"{uc_schema}.{table}"))
else:
    print("Skipping Genie resources because no Genie space is configured.")

print("Skipping Lakebase resource because Free Edition does not support Lakebase database instances.")

input_example = {
    "input": [
        {"role": "user", "content": "How does Kyle Freeland pitch to Freddie Freeman?"}
    ]
}

os.environ["ATBAT_MLFLOW_LOG_MODEL_DRY_RUN"] = "true"
try:
    with mlflow.start_run():
        logged_agent_info = mlflow.pyfunc.log_model(
            name="agent",
            python_model='agent_with_skills.py',
            input_example=input_example,
            resources=resources,
            pip_requirements=[
                "databricks-openai",
                "backoff",
                f"databricks-connect=={get_distribution('databricks-connect').version}",
                f"databricks-langchain=={get_distribution('databricks-langchain').version}",
                f"langgraph=={get_distribution('langgraph').version}",
                "langgraph-checkpoint-postgres",
                "psycopg[binary,pool]",
                "databricks-mcp",
            ],
        )
        print(f"Logged model: {logged_agent_info.model_uri}")
finally:
    os.environ.pop("ATBAT_MLFLOW_LOG_MODEL_DRY_RUN", None)


In [ ]:
# Register to Unity Catalog
mlflow.set_registry_uri("databricks-uc")

uc_registered_model_info = mlflow.register_model(
    model_uri=logged_agent_info.model_uri,
    name=UC_MODEL_NAME
)
print(f"Registered model: {UC_MODEL_NAME} version {uc_registered_model_info.version}")

In [ ]:
# Deploy the agent
from databricks import agents

# Build environment variables. Keep skills in the UC volume instead of packing
# them into an environment variable, which can exceed Databricks serving limits.
config_payload = Path("config/atbat_assistant.json").read_text()

skills_path = CONFIG["skills"]["gepa_volume_path"]
print(f"Skills will be loaded at runtime from UC volume: {skills_path}")

environment_vars = {
    "ATBAT_ASSISTANT_CONFIG_JSON": config_payload,
    "MLFLOW_TRACKING_URI": "databricks",
}

# Build authentication environment variables
auth_config = CONFIG["prompt_registry_auth"]
if auth_config.get("databricks_host"):
    if auth_config.get("use_oauth", True):
        environment_vars.update({
            "DATABRICKS_HOST": auth_config["databricks_host"],
            "DATABRICKS_CLIENT_ID": f"{{{{secrets/{auth_config['secret_scope_name']}/{auth_config['oauth_client_id_key']}}}}}",
            "DATABRICKS_CLIENT_SECRET": f"{{{{secrets/{auth_config['secret_scope_name']}/{auth_config['oauth_client_secret_key']}}}}}",
        })
        print(f"Using OAuth authentication (scope: {auth_config['secret_scope_name']})")
        print(f"MLflow tracking URI set to 'databricks' - traces will go to experiment {CONFIG['mlflow']['experiment_id']}")
    else:
        environment_vars.update({
            "DATABRICKS_HOST": auth_config["databricks_host"],
            "DATABRICKS_TOKEN": f"{{{{secrets/{auth_config['secret_scope_name']}/{auth_config.get('pat_key', 'pat')}}}}}",
        })
        print(f"Using PAT authentication (scope: {auth_config['secret_scope_name']})")
else:
    print("WARNING: DATABRICKS_HOST not configured - Prompt Registry and MLflow tracking may not work")

environment_vars["DISABLE_LAKEBASE"] = "true"
environment_vars["DISABLE_VECTOR_TOOLS"] = "true"
environment_vars["MLFLOW_HTTP_REQUEST_TIMEOUT"] = "300"
environment_vars["MLFLOW_HTTP_REQUEST_MAX_RETRIES"] = "1"
print("Lakebase disabled for deployment because Free Edition does not support Lakebase database instances.")

agents.deploy(
    UC_MODEL_NAME,
    uc_registered_model_info.version,
    scale_to_zero=True,
    environment_vars=environment_vars,
)
